<a href="https://colab.research.google.com/github/NMinarecioglu/kizilirmak-drought-propagation/blob/main/Copula_and_Bayesian_Network_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 pip install pandas numpy openpyxl

In [ ]:
"""
Copula Analysis for Drought Events
Fits bivariate copulas to (Duration, Severity) and (Duration, Peak) pairs
for each Station / Index / Scale / Threshold combination.

Requirements:
    pip install pandas numpy matplotlib scipy statsmodels openpyxl

Output:
    copula_plots/   — scatter + copula density plots
    copula_results.xlsx — fitted parameters and goodness-of-fit
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgrid
from scipy import stats
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import kendalltau, spearmanr
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
INPUT_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Copula_Results"

# Filter settings
THRESHOLD  = -0.5          # Use this threshold for copula fitting
PAIRS      = [("Duration", "Severity"), ("Duration", "Peak"), ("Severity", "Peak")]
MIN_EVENTS = 10            # Minimum events required to fit a copula
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_all = pd.read_csv(INPUT_FILE, parse_dates=["Start", "End"])
df_all["Peak"] = df_all["Peak"].abs()   # Work with absolute peak values

df = df_all[df_all["Threshold"] == THRESHOLD].copy().reset_index(drop=True)
print(f"Total events at threshold={THRESHOLD}: {len(df)}")


# ===========================================================
# EMPIRICAL CDF (Weibull plotting position)
# ===========================================================
def empirical_cdf(x):
    n    = len(x)
    rank = stats.rankdata(x, method="average")
    return rank / (n + 1)   # Weibull formula


# ===========================================================
# COPULA FAMILIES
# ===========================================================

# --- Gaussian ---
def gaussian_copula_loglik(rho, u, v):
    """Log-likelihood of Gaussian copula."""
    eps = 1e-10
    u   = np.clip(u, eps, 1 - eps)
    v   = np.clip(v, eps, 1 - eps)
    x   = stats.norm.ppf(u)
    y   = stats.norm.ppf(v)
    det = 1 - rho ** 2
    ll  = -0.5 * np.log(det) - (rho**2 * (x**2 + y**2) - 2*rho*x*y) / (2*det)
    return np.sum(ll)

def fit_gaussian(u, v):
    tau, _ = kendalltau(u, v)
    rho0   = np.sin(np.pi * tau / 2)
    res    = minimize_scalar(
        lambda r: -gaussian_copula_loglik(r, u, v),
        bounds=(-0.999, 0.999), method="bounded"
    )
    rho = res.x
    ll  = gaussian_copula_loglik(rho, u, v)
    return {"family": "Gaussian", "param1": round(rho, 4), "param2": np.nan,
            "loglik": round(ll, 4)}

# --- Clayton ---
def clayton_copula_cdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    return np.maximum(u**(-theta) + v**(-theta) - 1, eps) ** (-1/theta)

def clayton_copula_pdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    A   = u**(-theta) + v**(-theta) - 1
    pdf = (1+theta) * (u*v)**(-(1+theta)) * A**(-(2+1/theta))
    return np.maximum(pdf, eps)

def fit_clayton(u, v):
    tau, _ = kendalltau(u, v)
    theta0 = max(2*tau / (1-tau), 0.01)
    res    = minimize_scalar(
        lambda t: -np.sum(np.log(np.maximum(clayton_copula_pdf(u, v, t), 1e-300))),
        bounds=(0.01, 50), method="bounded"
    )
    theta = res.x
    ll    = np.sum(np.log(np.maximum(clayton_copula_pdf(u, v, theta), 1e-300)))
    return {"family": "Clayton", "param1": round(theta, 4), "param2": np.nan,
            "loglik": round(ll, 4)}

# --- Gumbel ---
def gumbel_copula_cdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    A   = (-np.log(u))**theta + (-np.log(v))**theta
    return np.exp(-A**(1/theta))

def gumbel_copula_pdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    lu  = -np.log(u)
    lv  = -np.log(v)
    A   = lu**theta + lv**theta
    C   = np.exp(-A**(1/theta))
    pdf = C * (u*v)**(-1) * A**(-2+2/theta) * (lu*lv)**(theta-1) * \
          (1 + (theta-1)*A**(-1/theta))
    return np.maximum(pdf, eps)

def fit_gumbel(u, v):
    tau, _ = kendalltau(u, v)
    theta0 = max(1 / (1 - tau), 1.01)
    res    = minimize_scalar(
        lambda t: -np.sum(np.log(np.maximum(gumbel_copula_pdf(u, v, t), 1e-300))),
        bounds=(1.001, 50), method="bounded"
    )
    theta = res.x
    ll    = np.sum(np.log(np.maximum(gumbel_copula_pdf(u, v, theta), 1e-300)))
    return {"family": "Gumbel", "param1": round(theta, 4), "param2": np.nan,
            "loglik": round(ll, 4)}

# --- Frank ---
def frank_copula_pdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    et  = np.exp(-theta)
    eu  = np.exp(-theta*u)
    ev  = np.exp(-theta*v)
    num = -theta * (et-1) * eu * ev
    den = ((et-1) + (eu-1)*(ev-1))**2
    pdf = num / np.maximum(den, eps)
    return np.maximum(np.abs(pdf), eps)

def fit_frank(u, v):
    res = minimize_scalar(
        lambda t: -np.sum(np.log(np.maximum(frank_copula_pdf(u, v, t), 1e-300))),
        bounds=(-50, 50), method="bounded"
    )
    theta = res.x
    ll    = np.sum(np.log(np.maximum(frank_copula_pdf(u, v, theta), 1e-300)))
    return {"family": "Frank", "param1": round(theta, 4), "param2": np.nan,
            "loglik": round(ll, 4)}


# ===========================================================
# SELECT BEST COPULA (AIC)
# ===========================================================
def select_best_copula(u, v):
    results = []
    for fit_fn in [fit_gaussian, fit_clayton, fit_gumbel, fit_frank]:
        try:
            r   = fit_fn(u, v)
            k   = 1   # all have 1 parameter
            aic = 2*k - 2*r["loglik"]
            r["AIC"] = round(aic, 4)
            results.append(r)
        except Exception as e:
            pass
    if not results:
        return None
    results.sort(key=lambda x: x["AIC"])
    return results   # sorted best first


# ===========================================================
# JOINT EXCEEDANCE PROBABILITY
# ===========================================================
def joint_exceed_prob(u, v, copula_info):
    """P(U > u_med, V > v_med) using best copula."""
    u_med = np.median(u)
    v_med = np.median(v)
    family = copula_info["family"]
    p      = copula_info["param1"]
    eps    = 1e-10

    if family == "Gaussian":
        from scipy.stats import multivariate_normal
        rho   = p
        cov   = [[1, rho], [rho, 1]]
        x_med = stats.norm.ppf(u_med)
        y_med = stats.norm.ppf(v_med)
        C_med = multivariate_normal.cdf([x_med, y_med], cov=cov)
    elif family == "Clayton":
        C_med = clayton_copula_cdf(np.array([u_med]), np.array([v_med]), p)[0]
    elif family == "Gumbel":
        C_med = gumbel_copula_cdf(np.array([u_med]), np.array([v_med]), p)[0]
    elif family == "Frank":
        et  = np.exp(-p)
        eu  = np.exp(-p*u_med)
        ev  = np.exp(-p*v_med)
        C_med = -1/p * np.log(1 + (eu-1)*(ev-1)/np.maximum(et-1, eps))
    else:
        C_med = u_med * v_med

    # P(U>u_med, V>v_med) = 1 - u_med - v_med + C(u_med, v_med)
    joint_p = 1 - u_med - v_med + C_med
    return round(float(joint_p), 4)


# ===========================================================
# PLOT: Scatter + Copula density
# ===========================================================
def plot_copula(u, v, x_raw, y_raw, best, station, idx, scale, pair, out_path):
    x_lab, y_lab = pair
    fig = plt.figure(figsize=(13, 5))
    fig.patch.set_facecolor("white")
    gs  = mgrid.GridSpec(1, 3, figure=fig, wspace=0.35)

    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[2])

    # --- Raw scatter ---
    ax1.scatter(x_raw, y_raw, alpha=0.6, s=20, color="#1f6f2e", edgecolors="white", lw=0.3)
    ax1.set_xlabel(x_lab, fontsize=10)
    ax1.set_ylabel(y_lab, fontsize=10)
    ax1.set_title("Observed Data", fontsize=10, fontweight="bold")
    ax1.grid(linestyle="--", alpha=0.4)

    # --- Uniform margin scatter ---
    ax2.scatter(u, v, alpha=0.6, s=20, color="#084594", edgecolors="white", lw=0.3)
    ax2.set_xlabel(f"F({x_lab})", fontsize=10)
    ax2.set_ylabel(f"F({y_lab})", fontsize=10)
    ax2.set_title("Probability Integral Transform", fontsize=10, fontweight="bold")
    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
    ax2.grid(linestyle="--", alpha=0.4)

    # --- Copula density contour ---
    ug, vg = np.meshgrid(np.linspace(0.01, 0.99, 80),
                          np.linspace(0.01, 0.99, 80))
    family = best["family"]
    p      = best["param1"]

    try:
        if family == "Gaussian":
            rho  = p
            xg   = stats.norm.ppf(ug)
            yg   = stats.norm.ppf(vg)
            det  = 1 - rho**2
            dens = np.exp(-0.5*(rho**2*(xg**2+yg**2)-2*rho*xg*yg)/det) / np.sqrt(det)
        elif family == "Clayton":
            dens = (1+p)*(ug*vg)**(-(1+p)) * \
                   np.maximum(ug**(-p)+vg**(-p)-1, 1e-10)**(-(2+1/p))
        elif family == "Gumbel":
            dens = gumbel_copula_pdf(ug, vg, p)
        elif family == "Frank":
            dens = frank_copula_pdf(ug, vg, p)
        else:
            dens = np.ones_like(ug)

        dens = np.where(np.isfinite(dens) & (dens > 0), dens, 1e-10)
        cf   = ax3.contourf(ug, vg, dens, levels=15, cmap="RdYlGn_r")
        ax3.contour(ug, vg, dens, levels=8, colors="black", linewidths=0.4, alpha=0.5)
        plt.colorbar(cf, ax=ax3, label="Density", pad=0.02)
    except Exception:
        ax3.text(0.5, 0.5, "Density\nunavailable", ha="center", va="center",
                 transform=ax3.transAxes)

    ax3.scatter(u, v, alpha=0.4, s=10, color="white", edgecolors="black", lw=0.4)
    ax3.set_xlabel(f"F({x_lab})", fontsize=10)
    ax3.set_ylabel(f"F({y_lab})", fontsize=10)
    ax3.set_title(f"{family} Copula Density\nθ={p}  AIC={best['AIC']}",
                  fontsize=10, fontweight="bold")
    ax3.set_xlim(0, 1); ax3.set_ylim(0, 1)

    tau, _  = kendalltau(u, v)
    rho_s,_ = spearmanr(u, v)
    fig.suptitle(
        f"{station} — {idx}-{scale}  |  {x_lab} vs {y_lab}\n"
        f"Kendall τ={tau:.3f}  Spearman ρ={rho_s:.3f}  "
        f"Best: {family} (AIC={best['AIC']})",
        fontsize=11, fontweight="bold", y=1.02
    )

    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()


# ===========================================================
# MAIN LOOP
# ===========================================================
print("=" * 60)
print("  Copula Analysis — Starting")
print("=" * 60 + "\n")

records = []

for station in sorted(df["Station"].unique()):
    for idx in ["SPI", "SDI", "RDI"]:
        for scale in [3, 6, 12]:
            sub = df[
                (df["Station"] == station) &
                (df["Index"]   == idx) &
                (df["Scale"]   == scale)
            ].dropna(subset=["Duration", "Severity", "Peak"]).copy()

            if len(sub) < MIN_EVENTS:
                print(f"  [SKIP] {station} {idx}-{scale}: only {len(sub)} events")
                continue

            print(f">>> {station} {idx}-{scale}  ({len(sub)} events)")

            for pair in PAIRS:
                x_col, y_col = pair
                x_raw = sub[x_col].values.astype(float)
                y_raw = sub[y_col].values.astype(float)

                # Uniform margins (empirical CDF)
                u = empirical_cdf(x_raw)
                v = empirical_cdf(y_raw)

                # Kendall tau
                tau, tau_p = kendalltau(x_raw, y_raw)

                # Fit all copulas and select best
                all_fits = select_best_copula(u, v)
                if all_fits is None:
                    continue
                best = all_fits[0]

                # Joint exceedance probability
                jp = joint_exceed_prob(u, v, best)

                # Save plot
                tag      = f"{station}_{idx}-{scale}_{x_col}_vs_{y_col}"
                out_path = os.path.join(OUTPUT_DIR, f"Copula_{tag}.png")
                plot_copula(u, v, x_raw, y_raw, best,
                            station, idx, scale, pair, out_path)

                # Record
                rec = {
                    "Station"     : station,
                    "Index"       : idx,
                    "Scale"       : scale,
                    "Threshold"   : THRESHOLD,
                    "Pair"        : f"{x_col}-{y_col}",
                    "N_Events"    : len(sub),
                    "Kendall_tau" : round(tau, 4),
                    "Tau_pvalue"  : round(tau_p, 4),
                    "Best_Copula" : best["family"],
                    "Param"       : best["param1"],
                    "AIC_Best"    : best["AIC"],
                    "LogLik"      : best["loglik"],
                    "Joint_Exceed_Prob": jp,
                }
                # Add all family AICs
                for r in all_fits:
                    rec[f"AIC_{r['family']}"] = r["AIC"]

                records.append(rec)
                print(f"    {x_col} vs {y_col}: Best={best['family']}  "
                      f"τ={tau:.3f}  AIC={best['AIC']}")

            print()

# ===========================================================
# SAVE RESULTS
# ===========================================================
df_res = pd.DataFrame(records)
out_xlsx = os.path.join(OUTPUT_DIR, "copula_results.xlsx")
out_csv  = os.path.join(OUTPUT_DIR, "copula_results.csv")
df_res.to_excel(out_xlsx, index=False)
df_res.to_csv(out_csv,   index=False)

print("=" * 60)
print(f"  Total fits : {len(df_res)}")
print(f"  Saved → {out_xlsx}")
print(f"  Saved → {out_csv}")
print("=" * 60)

print("\nBest copula family distribution:")
print(df_res["Best_Copula"].value_counts().to_string())


In [ ]:
"""
Bayesian Network Analysis for Drought Characteristics
Uses discretized drought features (Duration, Severity, Peak) to build
a Bayesian Network and compute conditional probabilities.

Requirements:
    pip install pandas numpy matplotlib scipy pgmpy openpyxl seaborn

Output:
    Bayesian_Network/ folder:
        - bn_results.xlsx       : conditional probability tables
        - BN_Structure_*.png    : network structure plots
        - BN_Heatmap_*.png      : conditional probability heatmaps
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE  = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
COPULA_FILE  = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Copula_Results/copula_results.xlsx"
OUTPUT_DIR   = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Bayesian_Network"

THRESHOLD    = -0.5       # Same threshold used in copula
SCALE        = 6          # Focus scale (change to 3 or 12 as needed)
INDICES      = ["SPI", "SDI", "RDI"]

# Discretization thresholds (percentile-based)
# Duration  : Short / Moderate / Long
# Severity  : Low / Moderate / High
# Peak      : Mild / Moderate / Extreme
PERCENTILES  = [33, 67]   # Tercile split
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------
# Load data
# ----------------------------------------------------------
df_ev = pd.read_csv(EVENTS_FILE, parse_dates=["Start", "End"])
df_ev["Peak"] = df_ev["Peak"].abs()
df_ev = df_ev[df_ev["Threshold"] == THRESHOLD].copy()

df_cop = pd.read_excel(COPULA_FILE)

print(f"Events loaded : {len(df_ev)}")
print(f"Copula results: {len(df_cop)}\n")


# ----------------------------------------------------------
# Discretize continuous variables into 3 categories
# ----------------------------------------------------------
def discretize(series, labels):
    p1, p2 = np.percentile(series, PERCENTILES)
    cats = pd.cut(series, bins=[-np.inf, p1, p2, np.inf],
                  labels=labels, include_lowest=True)
    return cats, p1, p2

DUR_LABELS = ["Short", "Moderate", "Long"]
SEV_LABELS = ["Low",   "Moderate", "High"]
PEK_LABELS = ["Mild",  "Moderate", "Extreme"]


# ----------------------------------------------------------
# Conditional Probability Tables (manual Bayesian Network)
# Structure: Duration → Severity → Peak
#            Duration → Peak
# ----------------------------------------------------------
def compute_cpt(df_sub):
    """
    Computes P(Severity | Duration) and P(Peak | Severity, Duration)
    Returns dict of DataFrames.
    """
    results = {}

    # P(Duration)
    p_dur = df_sub["D_cat"].value_counts(normalize=True).reindex(DUR_LABELS, fill_value=0)
    results["P_Duration"] = p_dur.round(4)

    # P(Severity | Duration)
    cpt_sev = pd.crosstab(df_sub["D_cat"], df_sub["S_cat"], normalize="index")
    cpt_sev = cpt_sev.reindex(index=DUR_LABELS, columns=SEV_LABELS, fill_value=0).round(4)
    results["P_Severity_given_Duration"] = cpt_sev

    # P(Peak | Severity)
    cpt_pek = pd.crosstab(df_sub["S_cat"], df_sub["P_cat"], normalize="index")
    cpt_pek = cpt_pek.reindex(index=SEV_LABELS, columns=PEK_LABELS, fill_value=0).round(4)
    results["P_Peak_given_Severity"] = cpt_pek

    # P(Peak | Duration, Severity)  — joint conditioning
    cpt_joint = df_sub.groupby(["D_cat", "S_cat"])["P_cat"] \
                      .value_counts(normalize=True) \
                      .unstack(fill_value=0) \
                      .reindex(columns=PEK_LABELS, fill_value=0).round(4)
    results["P_Peak_given_Duration_Severity"] = cpt_joint

    # Joint exceedance: P(D=Long, S=High, P=Extreme)
    n = len(df_sub)
    p_extreme = ((df_sub["D_cat"]=="Long") & (df_sub["S_cat"]=="High") &
                 (df_sub["P_cat"]=="Extreme")).sum() / n
    results["P_Extreme_Joint"] = round(p_extreme, 4)

    # Trigger probabilities: P(S=High | D=Long)
    sub_long = df_sub[df_sub["D_cat"] == "Long"]
    p_sev_high_given_long = (sub_long["S_cat"] == "High").mean() if len(sub_long) > 0 else np.nan
    results["P_SevHigh_given_DurLong"] = round(p_sev_high_given_long, 4)

    # P(Peak=Extreme | S=High)
    sub_high = df_sub[df_sub["S_cat"] == "High"]
    p_ext_given_high = (sub_high["P_cat"] == "Extreme").mean() if len(sub_high) > 0 else np.nan
    results["P_PeakExtreme_given_SevHigh"] = round(p_ext_given_high, 4)

    return results


# ----------------------------------------------------------
# Plot: Network Structure
# ----------------------------------------------------------
def plot_bn_structure(station, idx, scale, cpt_results, out_path):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor("white")

    # Colors per index
    colors = {"SPI": "#1f6f2e", "SDI": "#084594", "RDI": "#8c2d04"}
    col    = colors.get(idx, "gray")

    # --- Panel 1: P(Severity | Duration) ---
    ax = axes[0]
    cpt = cpt_results["P_Severity_given_Duration"]
    x   = np.arange(len(DUR_LABELS))
    w   = 0.25
    shades = ["#a8d5a2", "#4caf50", "#1b5e20"]
    for i, sev in enumerate(SEV_LABELS):
        vals = cpt[sev].values if sev in cpt.columns else np.zeros(3)
        ax.bar(x + i*w, vals, w, label=sev, color=shades[i], edgecolor="white")
    ax.set_xticks(x + w)
    ax.set_xticklabels(DUR_LABELS, fontsize=9)
    ax.set_ylabel("Probability", fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title("P(Severity | Duration)", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, title="Severity")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    # --- Panel 2: P(Peak | Severity) ---
    ax = axes[1]
    cpt = cpt_results["P_Peak_given_Severity"]
    shades2 = ["#ffcc80", "#ef6c00", "#b71c1c"]
    for i, pek in enumerate(PEK_LABELS):
        vals = cpt[pek].values if pek in cpt.columns else np.zeros(3)
        ax.bar(x + i*w, vals, w, label=pek, color=shades2[i], edgecolor="white")
    ax.set_xticks(x + w)
    ax.set_xticklabels(SEV_LABELS, fontsize=9)
    ax.set_ylabel("Probability", fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title("P(Peak | Severity)", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, title="Peak")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    # --- Panel 3: Key probabilities (trigger thresholds) ---
    ax = axes[2]
    ax.axis("off")

    p_ext   = cpt_results["P_Extreme_Joint"]
    p_sh_dl = cpt_results["P_SevHigh_given_DurLong"]
    p_pe_sh = cpt_results["P_PeakExtreme_given_SevHigh"]
    p_dur   = cpt_results["P_Duration"]

    table_data = [
        ["P(Duration=Short)",            f"{p_dur.get('Short', 0):.3f}"],
        ["P(Duration=Moderate)",         f"{p_dur.get('Moderate', 0):.3f}"],
        ["P(Duration=Long)",             f"{p_dur.get('Long', 0):.3f}"],
        ["─────────────────────", "──────"],
        ["P(Sev=High | Dur=Long)",       f"{p_sh_dl:.3f}" if not np.isnan(p_sh_dl) else "N/A"],
        ["P(Peak=Extreme | Sev=High)",   f"{p_pe_sh:.3f}" if not np.isnan(p_pe_sh) else "N/A"],
        ["─────────────────────", "──────"],
        ["P(Long, High, Extreme)",       f"{p_ext:.3f}"],
    ]

    tbl = ax.table(
        cellText=table_data,
        colLabels=["Probability Statement", "Value"],
        loc="center", cellLoc="left"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.3, 1.6)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(col)
            cell.set_text_props(color="white", fontweight="bold")
        elif r in [4, 5, 7]:
            cell.set_facecolor("#fff3e0")
    ax.set_title("Trigger Probabilities", fontsize=10, fontweight="bold")

    fig.suptitle(
        f"Bayesian Network — {station}  {idx}-{scale}\n"
        f"Structure: Duration → Severity → Peak",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()


# ----------------------------------------------------------
# Plot: Conditional Probability Heatmap
# ----------------------------------------------------------
def plot_heatmap(station, idx, scale, cpt_results, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.patch.set_facecolor("white")

    cpt1 = cpt_results["P_Severity_given_Duration"]
    cpt2 = cpt_results["P_Peak_given_Severity"]

    sns.heatmap(cpt1, ax=axes[0], annot=True, fmt=".3f",
                cmap="YlOrRd", vmin=0, vmax=1,
                linewidths=0.5, linecolor="white",
                cbar_kws={"label": "Probability"})
    axes[0].set_title("P(Severity | Duration)", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Severity Category", fontsize=9)
    axes[0].set_ylabel("Duration Category", fontsize=9)

    sns.heatmap(cpt2, ax=axes[1], annot=True, fmt=".3f",
                cmap="YlOrRd", vmin=0, vmax=1,
                linewidths=0.5, linecolor="white",
                cbar_kws={"label": "Probability"})
    axes[1].set_title("P(Peak | Severity)", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Peak Category", fontsize=9)
    axes[1].set_ylabel("Severity Category", fontsize=9)

    fig.suptitle(
        f"Conditional Probability Heatmaps — {station}  {idx}-{scale}",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close()


# ----------------------------------------------------------
# Main loop
# ----------------------------------------------------------
print("=" * 60)
print("  Bayesian Network Analysis — Starting")
print("=" * 60 + "\n")

all_records = []

stations = sorted(df_ev["Station"].unique())

for station in stations:
    for idx in INDICES:
        for scale in [3, 6, 12]:
            sub = df_ev[
                (df_ev["Station"] == station) &
                (df_ev["Index"]   == idx) &
                (df_ev["Scale"]   == scale)
            ].dropna(subset=["Duration", "Severity", "Peak"]).copy()

            if len(sub) < 10:
                print(f"  [SKIP] {station} {idx}-{scale}: only {len(sub)} events")
                continue

            # Discretize
            sub["D_cat"], d1, d2 = discretize(sub["Duration"], DUR_LABELS)
            sub["S_cat"], s1, s2 = discretize(sub["Severity"], SEV_LABELS)
            sub["P_cat"], p1, p2 = discretize(sub["Peak"],     PEK_LABELS)

            print(f">>> {station} {idx}-{scale}  ({len(sub)} events)")
            print(f"    Duration thresholds : {d1:.1f} / {d2:.1f} months")
            print(f"    Severity thresholds : {s1:.2f} / {s2:.2f}")
            print(f"    Peak thresholds     : {p1:.2f} / {p2:.2f}")

            # CPT
            cpt_res = compute_cpt(sub)
            print(f"    P(Sev=High | Dur=Long)      = {cpt_res['P_SevHigh_given_DurLong']:.3f}")
            print(f"    P(Peak=Extreme | Sev=High)  = {cpt_res['P_PeakExtreme_given_SevHigh']:.3f}")
            print(f"    P(Long & High & Extreme)    = {cpt_res['P_Extreme_Joint']:.3f}\n")

            # Plots
            tag = f"{station}_{idx}-{scale}"
            plot_bn_structure(
                station, idx, scale, cpt_res,
                os.path.join(OUTPUT_DIR, f"BN_Structure_{tag}.png")
            )
            plot_heatmap(
                station, idx, scale, cpt_res,
                os.path.join(OUTPUT_DIR, f"BN_Heatmap_{tag}.png")
            )

            # Record summary
            p_dur = cpt_res["P_Duration"]
            all_records.append({
                "Station"                     : station,
                "Index"                       : idx,
                "Scale"                       : scale,
                "N_Events"                    : len(sub),
                "Dur_Short_thresh"            : round(d1, 2),
                "Dur_Long_thresh"             : round(d2, 2),
                "Sev_Low_thresh"              : round(s1, 2),
                "Sev_High_thresh"             : round(s2, 2),
                "Peak_Mild_thresh"            : round(p1, 2),
                "Peak_Extreme_thresh"         : round(p2, 2),
                "P_Duration_Short"            : p_dur.get("Short", 0),
                "P_Duration_Moderate"         : p_dur.get("Moderate", 0),
                "P_Duration_Long"             : p_dur.get("Long", 0),
                "P_SevHigh_given_DurLong"     : cpt_res["P_SevHigh_given_DurLong"],
                "P_PeakExtreme_given_SevHigh" : cpt_res["P_PeakExtreme_given_SevHigh"],
                "P_Extreme_Joint"             : cpt_res["P_Extreme_Joint"],
            })

# ----------------------------------------------------------
# Save results
# ----------------------------------------------------------
df_bn = pd.DataFrame(all_records)
out_xlsx = os.path.join(OUTPUT_DIR, "bn_results.xlsx")
out_csv  = os.path.join(OUTPUT_DIR, "bn_results.csv")
df_bn.to_excel(out_xlsx, index=False)
df_bn.to_csv(out_csv,   index=False)

print("=" * 60)
print(f"  Total combinations : {len(df_bn)}")
print(f"  Saved → {out_xlsx}")
print("=" * 60)

# Summary: highest joint extreme probabilities
print("\nTop 10 — Highest P(Long & High & Extreme):")
print(df_bn.nlargest(10, "P_Extreme_Joint")[
    ["Station", "Index", "Scale", "N_Events",
     "P_SevHigh_given_DurLong",
     "P_PeakExtreme_given_SevHigh",
     "P_Extreme_Joint"]
].to_string(index=False))

In [ ]:
pip install pandas numpy matplotlib scipy statsmodels openpyxl seaborn

In [ ]:
"""
Response Probability Contour Plots (Figure 9-10) and
Trigger Threshold Analysis (Figure 13-14)

For each station: P(SDI duration | SPI duration) and P(SDI severity | SPI severity)
Trigger threshold: SPI value needed to induce SDI <= -0.5

Requirements:
    pip install pandas numpy matplotlib scipy statsmodels openpyxl seaborn

Output:
    Response_Probability/ — contour plots
    Trigger_Threshold/    — threshold plots and table
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from scipy.optimize import minimize_scalar
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_RESP = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Response_Probability"
OUTPUT_TRIG = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Trigger_Threshold"

THRESHOLD   = -0.5      # Run theory threshold used
SCALE       = 6         # Primary scale to focus on (also loops 3,6,12)
STATIONS    = ["Kastamonu", "Sivas", "Kayseri", "Yozgat", "Kirikkale"]

# Trigger threshold settings
SPI_START   = -0.5
SPI_END     = -3.0
SPI_STEP    = -0.1
SDI_TRIGGER = -0.5      # Agricultural/hydrological drought threshold
# ============================================================

os.makedirs(OUTPUT_RESP, exist_ok=True)
os.makedirs(OUTPUT_TRIG, exist_ok=True)

df_all = pd.read_csv(EVENTS_FILE, parse_dates=["Start", "End"])
df_all["Peak"] = df_all["Peak"].abs()
df_ev = df_all[df_all["Threshold"] == THRESHOLD].copy()

print(f"Events loaded: {len(df_ev)}")


# ===========================================================
# COPULA HELPERS
# ===========================================================
def empirical_cdf(x):
    n    = len(x)
    rank = stats.rankdata(x, method="average")
    return rank / (n + 1)

def fit_best_marginal(x):
    """Fit best marginal from: Gamma, LogNorm, GEV, Exponential."""
    x = np.array(x)
    x = x[x > 0]
    best_ks, best_dist, best_params = np.inf, None, None
    candidates = [
        ("gamma",   stats.gamma),
        ("lognorm", stats.lognorm),
        ("genextreme", stats.genextreme),
        ("expon",   stats.expon),
        ("weibull_min", stats.weibull_min),
    ]
    for name, dist in candidates:
        try:
            params = dist.fit(x, floc=0)
            ks, _  = stats.kstest(x, dist.cdf, args=params)
            if ks < best_ks:
                best_ks = ks
                best_dist = dist
                best_params = params
        except Exception:
            pass
    return best_dist, best_params

def marginal_cdf(x_val, dist, params):
    return np.clip(dist.cdf(x_val, *params), 1e-6, 1 - 1e-6)

# Frank copula PDF and CDF
def frank_cdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    et  = np.exp(-theta)
    eu  = np.exp(-theta * u)
    ev  = np.exp(-theta * v)
    val = -1/theta * np.log(1 + (eu-1)*(ev-1) / np.maximum(et-1, eps))
    return np.clip(val, eps, 1-eps)

def frank_pdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    et  = np.exp(-theta)
    eu  = np.exp(-theta*u)
    ev  = np.exp(-theta*v)
    num = -theta * (et-1) * eu * ev
    den = ((et-1) + (eu-1)*(ev-1))**2
    return np.maximum(np.abs(num / np.maximum(den, eps)), eps)

def gumbel_cdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    A   = (-np.log(u))**theta + (-np.log(v))**theta
    return np.exp(-A**(1/theta))

def clayton_cdf(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    return np.maximum(u**(-theta) + v**(-theta) - 1, eps)**(-1/theta)

def gaussian_cdf(u, v, rho):
    from scipy.stats import multivariate_normal
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    x   = stats.norm.ppf(u)
    y   = stats.norm.ppf(v)
    result = np.zeros_like(u)
    for i in range(len(u.ravel())):
        ui = u.ravel()[i]
        vi = v.ravel()[i]
        xi = stats.norm.ppf(ui)
        yi = stats.norm.ppf(vi)
        result.ravel()[i] = multivariate_normal.cdf(
            [xi, yi], mean=[0,0], cov=[[1, rho],[rho, 1]]
        )
    return result.reshape(u.shape)

def fit_copula(u, v):
    """Fit Frank, Gumbel, Clayton, Gaussian; return best by AIC."""
    from scipy.stats import kendalltau
    tau, _ = kendalltau(u, v)

    results = []

    # Frank
    try:
        res = minimize_scalar(
            lambda t: -np.sum(np.log(np.maximum(frank_pdf(u, v, t), 1e-300))),
            bounds=(-50, 50), method="bounded"
        )
        ll  = -res.fun
        aic = 2 - 2*ll
        results.append(("Frank",    res.x,  np.nan, ll, aic, frank_cdf))
    except Exception: pass

    # Gumbel (theta >= 1)
    try:
        theta0 = max(1/(1-tau), 1.01) if tau < 1 else 2.0
        res = minimize_scalar(
            lambda t: -np.sum(np.log(np.maximum(
                gumbel_copula_pdf_safe(u, v, t), 1e-300))),
            bounds=(1.001, 50), method="bounded"
        )
        ll  = -res.fun
        aic = 2 - 2*ll
        results.append(("Gumbel",   res.x, np.nan, ll, aic, gumbel_cdf))
    except Exception: pass

    # Clayton (theta > 0)
    try:
        res = minimize_scalar(
            lambda t: -np.sum(np.log(np.maximum(
                clayton_pdf_safe(u, v, t), 1e-300))),
            bounds=(0.01, 50), method="bounded"
        )
        ll  = -res.fun
        aic = 2 - 2*ll
        results.append(("Clayton",  res.x, np.nan, ll, aic, clayton_cdf))
    except Exception: pass

    # Gaussian
    try:
        rho0 = np.sin(np.pi * tau / 2)
        res  = minimize_scalar(
            lambda r: -gaussian_ll(u, v, r),
            bounds=(-0.999, 0.999), method="bounded"
        )
        ll  = -res.fun
        aic = 2 - 2*ll
        results.append(("Gaussian", res.x, np.nan, ll, aic, gaussian_cdf))
    except Exception: pass

    if not results:
        return None
    results.sort(key=lambda x: x[4])
    return results[0]   # (family, param, _, ll, aic, cdf_fn)

def gumbel_copula_pdf_safe(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    lu  = -np.log(u); lv = -np.log(v)
    A   = lu**theta + lv**theta
    C   = np.exp(-A**(1/theta))
    pdf = C*(u*v)**(-1)*A**(-2+2/theta)*(lu*lv)**(theta-1)*(1+(theta-1)*A**(-1/theta))
    return np.maximum(pdf, eps)

def clayton_pdf_safe(u, v, theta):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    A   = u**(-theta) + v**(-theta) - 1
    pdf = (1+theta)*(u*v)**(-(1+theta))*np.maximum(A, eps)**(-(2+1/theta))
    return np.maximum(pdf, eps)

def gaussian_ll(u, v, rho):
    eps = 1e-10
    u   = np.clip(u, eps, 1-eps)
    v   = np.clip(v, eps, 1-eps)
    x   = stats.norm.ppf(u)
    y   = stats.norm.ppf(v)
    det = 1 - rho**2
    ll  = -0.5*np.log(det) - (rho**2*(x**2+y**2) - 2*rho*x*y)/(2*det)
    return np.sum(ll)


# ===========================================================
# CONDITIONAL PROBABILITY:  P(V <= v | U = u)
# dC/du = partial derivative of C w.r.t. u
# ===========================================================
def conditional_prob(u_val, v_vals, family, param, cdf_fn):
    """P(V <= v | U = u_val) via numerical derivative."""
    h  = 1e-4
    u1 = np.clip(u_val + h, 1e-6, 1-1e-6)
    u2 = np.clip(u_val - h, 1e-6, 1-1e-6)
    v  = np.clip(v_vals, 1e-6, 1-1e-6)

    if family == "Gaussian":
        c1 = gaussian_cdf(np.full_like(v, u1), v, param)
        c2 = gaussian_cdf(np.full_like(v, u2), v, param)
    elif family == "Frank":
        c1 = frank_cdf(np.full_like(v, u1), v, param)
        c2 = frank_cdf(np.full_like(v, u2), v, param)
    elif family == "Gumbel":
        c1 = gumbel_cdf(np.full_like(v, u1), v, param)
        c2 = gumbel_cdf(np.full_like(v, u2), v, param)
    elif family == "Clayton":
        c1 = clayton_cdf(np.full_like(v, u1), v, param)
        c2 = clayton_cdf(np.full_like(v, u2), v, param)
    else:
        return v   # fallback: independence

    return np.clip((c1 - c2) / (u1 - u2), 0, 1)


# ===========================================================
# FIGURE 9-10: Response Probability Contour Plots
# ===========================================================
def plot_response_probability(variable="Duration", scale=6):
    """
    For each station: P(SDI_var <= y | SPI_var = x) contour plot.
    variable: "Duration" or "Severity"
    """
    fig, axes = plt.subplots(1, len(STATIONS), figsize=(18, 5), sharey=True)
    fig.patch.set_facecolor("white")

    # X axis: meteorological drought (SPI) variable values
    x_max = 12 if variable == "Duration" else 15
    y_max = 12 if variable == "Duration" else 15
    x_grid = np.linspace(0.5, x_max, 30)
    y_grid = np.linspace(0.5, y_max, 30)

    levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    for ax, station in zip(axes, STATIONS):
        # Get SPI and SDI events
        spi_ev = df_ev[(df_ev["Station"]==station) &
                       (df_ev["Index"]=="SPI") &
                       (df_ev["Scale"]==scale)][variable].dropna().values
        sdi_ev = df_ev[(df_ev["Station"]==station) &
                       (df_ev["Index"]=="SDI") &
                       (df_ev["Scale"]==scale)][variable].dropna().values

        if len(spi_ev) < 8 or len(sdi_ev) < 8:
            ax.text(0.5, 0.5, f"Insufficient\ndata\n({len(spi_ev)},{len(sdi_ev)})",
                    ha="center", va="center", transform=ax.transAxes, fontsize=9)
            ax.set_title(station, fontsize=10, fontweight="bold")
            continue

        # Fit marginal distributions
        dist_x, par_x = fit_best_marginal(spi_ev)
        dist_y, par_y = fit_best_marginal(sdi_ev)

        # Fit copula on combined data (use shorter array)
        n_min = min(len(spi_ev), len(sdi_ev))
        u = empirical_cdf(spi_ev[:n_min])
        v = empirical_cdf(sdi_ev[:n_min])
        cop = fit_copula(u, v)

        if cop is None:
            ax.text(0.5, 0.5, "Copula\nfit failed",
                    ha="center", va="center", transform=ax.transAxes)
            ax.set_title(station, fontsize=10, fontweight="bold")
            continue

        family, param = cop[0], cop[1]

        # Compute conditional probability grid
        prob_grid = np.zeros((len(y_grid), len(x_grid)))
        for j, xv in enumerate(x_grid):
            u_xv  = marginal_cdf(xv, dist_x, par_x)
            v_arr = np.array([marginal_cdf(yv, dist_y, par_y) for yv in y_grid])
            probs = conditional_prob(u_xv, v_arr, family, param, None)
            prob_grid[:, j] = probs

        # Plot
        cs = ax.contour(x_grid, y_grid, prob_grid,
                        levels=levels, colors=[
                            "#4575b4","#74add1","#abd9e9","#e0f3f8",
                            "#fee090","#fdae61","#f46d43","#d73027","#a50026"
                        ], linewidths=1.4)
        ax.clabel(cs, fmt="%.2f", fontsize=7, inline=True)

        ax.set_xlim(0, x_max)
        ax.set_ylim(0, y_max)
        ax.set_xlabel(f"SPI-{scale} {variable} (months)" if variable=="Duration"
                      else f"SPI-{scale} {variable}", fontsize=8)
        ax.set_title(f"{station}\n{family} (τ={stats.kendalltau(spi_ev[:n_min],sdi_ev[:n_min])[0]:.2f})",
                     fontsize=9, fontweight="bold")
        ax.grid(linestyle="--", alpha=0.3)

    axes[0].set_ylabel(f"SDI-{scale} {variable}", fontsize=9)

    fig.suptitle(
        f"Response Probability: P(SDI-{scale} {variable} ≤ y | SPI-{scale} {variable} = x)\n"
        f"Threshold = {THRESHOLD}  |  Contour values = conditional probability",
        fontsize=11, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_RESP, f"ResponseProb_{variable}_Scale{scale}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ===========================================================
# FIGURE 13: Trigger Threshold — P(SDI <= -0.5 | SPI in [x-0.1, x])
# ===========================================================
def compute_trigger_threshold(station, index_x="SPI", index_y="SDI", scale=6):
    """
    For each SPI interval [xi-0.1, xi], compute:
    P(SDI <= SDI_TRIGGER | xi-0.1 < SPI <= xi)
    = [C(F_SPI(xi), F_SDI(y)) - C(F_SPI(xi-0.1), F_SDI(y))] /
      [F_SPI(xi) - F_SPI(xi-0.1)]
    """
    spi_series = df_ev[(df_ev["Station"]==station) &
                       (df_ev["Index"]==index_x) &
                       (df_ev["Scale"]==scale)]["Severity"].dropna().values
    sdi_series = df_ev[(df_ev["Station"]==station) &
                       (df_ev["Index"]==index_y) &
                       (df_ev["Scale"]==scale)]["Severity"].dropna().values

    # Use raw monthly index values instead — need original series
    # For trigger threshold we work with the ORIGINAL SPI/SDI monthly values
    # Since we only have drought events, we approximate using severity as proxy
    # and compute P(SDI_severity > threshold | SPI_severity in range)

    if len(spi_series) < 8 or len(sdi_series) < 8:
        return None, None

    n = min(len(spi_series), len(sdi_series))
    spi_s = np.sort(spi_series[:n])
    sdi_s = sdi_series[:n]

    dist_x, par_x = fit_best_marginal(spi_s)
    dist_y, par_y = fit_best_marginal(np.abs(sdi_s))

    u = empirical_cdf(spi_s)
    v = empirical_cdf(np.abs(sdi_s))
    cop = fit_copula(u, v)
    if cop is None:
        return None, None

    family, param = cop[0], cop[1]

    # SPI range to iterate (use percentile-based range of observed values)
    spi_range = np.arange(
        max(spi_s.min(), 0.5),
        min(spi_s.max(), 20),
        0.5
    )

    # Threshold for SDI severity (use median as proxy)
    sdi_thr = np.percentile(np.abs(sdi_s), 50)

    probs = []
    x_vals = []
    for xi in spi_range:
        xi0 = max(xi - 0.5, 0.01)
        u2  = marginal_cdf(xi,  dist_x, par_x)
        u1  = marginal_cdf(xi0, dist_x, par_x)
        v_t = marginal_cdf(sdi_thr, dist_y, par_y)

        if family == "Frank":
            c2 = frank_cdf(np.array([u2]), np.array([v_t]), param)[0]
            c1 = frank_cdf(np.array([u1]), np.array([v_t]), param)[0]
        elif family == "Gumbel":
            c2 = gumbel_cdf(np.array([u2]), np.array([v_t]), param)[0]
            c1 = gumbel_cdf(np.array([u1]), np.array([v_t]), param)[0]
        elif family == "Clayton":
            c2 = clayton_cdf(np.array([u2]), np.array([v_t]), param)[0]
            c1 = clayton_cdf(np.array([u1]), np.array([v_t]), param)[0]
        else:  # Gaussian
            c2 = gaussian_cdf(np.array([u2]), np.array([v_t]), param)[0]
            c1 = gaussian_cdf(np.array([u1]), np.array([v_t]), param)[0]

        denom = u2 - u1
        if denom < 1e-8:
            continue
        p = (c2 - c1) / denom
        probs.append(np.clip(p, 0, 1))
        x_vals.append(xi)

    return np.array(x_vals), np.array(probs)


def plot_trigger_threshold(scale=6):
    """Figure 13-14: Trigger threshold curves for all stations."""
    colors = {"Kastamonu": "#1f77b4", "Sivas": "#ff7f0e",
              "Kayseri": "#2ca02c", "Yozgat": "#d62728",
              "Kirikkale": "#9467bd"}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor("white")

    pairs = [("SPI", "SDI"), ("SPI", "RDI")]
    axes_pairs = [axes[0], axes[1]]

    all_records = []

    for ax, (idx_x, idx_y) in zip(axes_pairs, pairs):
        for station in STATIONS:
            x_vals, probs = compute_trigger_threshold(
                station, idx_x, idx_y, scale
            )
            if x_vals is None:
                continue
            ax.plot(x_vals, probs, "-o", markersize=3,
                    color=colors[station], label=station, linewidth=1.8)

            # Find 50% trigger threshold
            if len(probs) > 0 and probs.max() >= 0.5:
                idx50 = np.argmin(np.abs(probs - 0.5))
                all_records.append({
                    "Station": station,
                    "Pair"   : f"{idx_x}-{idx_y}",
                    "Scale"  : scale,
                    "X_at_50pct": round(x_vals[idx50], 2),
                    "Max_prob"  : round(probs.max(), 3),
                })

        ax.axhline(0.5, color="black", linewidth=1.2,
                   linestyle="--", label="50% threshold")
        ax.set_xlabel(f"{idx_x}-{scale} Severity", fontsize=10)
        ax.set_ylabel(f"P({idx_y} ≤ median | {idx_x} in interval)", fontsize=9)
        ax.set_ylim(0, 1)
        ax.set_title(f"{idx_x} → {idx_y}  (Scale-{scale})",
                     fontsize=11, fontweight="bold")
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(linestyle="--", alpha=0.4)

    # Panel 3: Threshold summary box plot
    ax3 = axes[2]
    if all_records:
        df_rec = pd.DataFrame(all_records)
        for i, pair in enumerate(df_rec["Pair"].unique()):
            vals = df_rec[df_rec["Pair"]==pair]["X_at_50pct"].values
            ax3.boxplot(vals, positions=[i+1], widths=0.4,
                        patch_artist=True,
                        boxprops=dict(facecolor=["#74add1","#f46d43"][i], alpha=0.7))
        ax3.set_xticks([1, 2])
        ax3.set_xticklabels(df_rec["Pair"].unique(), fontsize=9)
        ax3.set_ylabel("SPI Severity at 50% Trigger", fontsize=9)
        ax3.set_title("Trigger Threshold Distribution\n(50% probability level)",
                      fontsize=10, fontweight="bold")
        ax3.grid(linestyle="--", alpha=0.4)

    fig.suptitle(
        f"Trigger Thresholds: Meteorological Drought → Hydrological Drought  (Scale-{scale})\n"
        f"Threshold = {THRESHOLD}  |  Dashed line = 50% response probability",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_TRIG, f"TriggerThreshold_Scale{scale}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")

    # Save table
    if all_records:
        df_rec = pd.DataFrame(all_records)
        df_rec.to_excel(os.path.join(OUTPUT_TRIG, "trigger_threshold_table.xlsx"), index=False)
        df_rec.to_csv(os.path.join(OUTPUT_TRIG,  "trigger_threshold_table.csv"),  index=False)
        print(f"\n  Trigger threshold table saved.")
        print(df_rec.to_string(index=False))


# ===========================================================
# FIGURE 14: Multi-level drought grade propagation probability
# Drought grades: Mild(-1,-0.5), Moderate(-1.5,-1), Severe(-2,-1.5), Extreme(<-2)
# ===========================================================
def plot_grade_propagation(scale=6):
    """
    P(SDI <= y | x1 < SPI <= x2) for 4 drought grades x 4 SDI levels.
    Uses event severity as proxy for SPI/SDI grade.
    """
    grades_spi = {
        "Mild"     : (0.5,  1.5),
        "Moderate" : (1.5,  3.0),
        "Severe"   : (3.0,  6.0),
        "Extreme"  : (6.0, 20.0),
    }
    grades_sdi = {
        "Mild"     : 1.5,
        "Moderate" : 3.0,
        "Severe"   : 6.0,
        "Extreme"  : 20.0,
    }

    fig, axes = plt.subplots(1, len(STATIONS), figsize=(18, 5), sharey=True)
    fig.patch.set_facecolor("white")

    colors_grade = {"Mild":"#74add1","Moderate":"#fdae61",
                    "Severe":"#f46d43","Extreme":"#a50026"}
    markers      = {"Mild":"o","Moderate":"s","Severe":"^","Extreme":"D"}

    for ax, station in zip(axes, STATIONS):
        spi_ev = df_ev[(df_ev["Station"]==station) &
                       (df_ev["Index"]=="SPI") &
                       (df_ev["Scale"]==scale)]["Severity"].dropna().values
        sdi_ev = df_ev[(df_ev["Station"]==station) &
                       (df_ev["Index"]=="SDI") &
                       (df_ev["Scale"]==scale)]["Severity"].dropna().values

        if len(spi_ev) < 8 or len(sdi_ev) < 8:
            ax.text(0.5, 0.5, "Insufficient\ndata",
                    ha="center", va="center", transform=ax.transAxes)
            ax.set_title(station, fontsize=10, fontweight="bold")
            continue

        n = min(len(spi_ev), len(sdi_ev))
        dist_x, par_x = fit_best_marginal(spi_ev[:n])
        dist_y, par_y = fit_best_marginal(sdi_ev[:n])
        u = empirical_cdf(spi_ev[:n])
        v = empirical_cdf(sdi_ev[:n])
        cop = fit_copula(u, v)
        if cop is None:
            continue
        family, param = cop[0], cop[1]

        for spi_grade, (lo, hi) in grades_spi.items():
            u2 = marginal_cdf(hi, dist_x, par_x)
            u1 = marginal_cdf(lo, dist_x, par_x)
            denom = u2 - u1
            if denom < 1e-8:
                continue

            sdi_levels = np.array([grades_sdi[g] for g in grades_sdi])
            probs = []
            for sdi_thr in sdi_levels:
                v_t = marginal_cdf(sdi_thr, dist_y, par_y)
                if family == "Frank":
                    c2 = frank_cdf(np.array([u2]), np.array([v_t]), param)[0]
                    c1 = frank_cdf(np.array([u1]), np.array([v_t]), param)[0]
                elif family == "Gumbel":
                    c2 = gumbel_cdf(np.array([u2]), np.array([v_t]), param)[0]
                    c1 = gumbel_cdf(np.array([u1]), np.array([v_t]), param)[0]
                elif family == "Clayton":
                    c2 = clayton_cdf(np.array([u2]), np.array([v_t]), param)[0]
                    c1 = clayton_cdf(np.array([u1]), np.array([v_t]), param)[0]
                else:
                    c2 = gaussian_cdf(np.array([u2]), np.array([v_t]), param)[0]
                    c1 = gaussian_cdf(np.array([u1]), np.array([v_t]), param)[0]
                probs.append(np.clip((c2-c1)/denom, 0, 1))

            ax.plot(list(grades_sdi.keys()), probs,
                    "-" + markers[spi_grade], color=colors_grade[spi_grade],
                    label=f"SPI {spi_grade}", linewidth=1.6, markersize=6)

        ax.set_ylim(0, 1)
        ax.set_xlabel("SDI Drought Grade", fontsize=8)
        ax.set_title(station, fontsize=10, fontweight="bold")
        ax.grid(linestyle="--", alpha=0.3)
        ax.legend(fontsize=7, loc="lower right")

    axes[0].set_ylabel("P(SDI ≤ grade | SPI grade)", fontsize=9)
    fig.suptitle(
        f"Probability of Meteorological Drought (SPI-{scale}) Inducing "
        f"Hydrological Drought (SDI-{scale})\n"
        f"16 drought grade combinations (4 SPI × 4 SDI levels)",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_TRIG, f"GradePropagation_Scale{scale}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ===========================================================
# MAIN
# ===========================================================
print("=" * 60)
print("  Response Probability & Trigger Threshold Analysis")
print("=" * 60 + "\n")

for scale in [3, 6, 12]:
    print(f"\n--- Scale: {scale} months ---")

    print("  [Fig 9] Duration response probability...")
    plot_response_probability("Duration", scale)

    print("  [Fig 10] Severity response probability...")
    plot_response_probability("Severity", scale)

    print("  [Fig 13] Trigger threshold curves...")
    plot_trigger_threshold(scale)

    print("  [Fig 14] Grade propagation probability...")
    plot_grade_propagation(scale)

print("\nAll plots completed!")
print(f"Response Prob  -> {OUTPUT_RESP}")
print(f"Trigger Thresh -> {OUTPUT_TRIG}")

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv",
    parse_dates=["Start","End"]
)
df = df[df["Threshold"] == -0.5]

for station in df["Station"].unique():
    for scale in [3, 6, 12]:
        spi = df[(df["Station"]==station) & (df["Index"]=="SPI") & (df["Scale"]==scale)][["Start","Duration","Severity"]].rename(columns={"Duration":"D_SPI","Severity":"S_SPI"})
        sdi = df[(df["Station"]==station) & (df["Index"]=="SDI") & (df["Scale"]==scale)][["Start","Duration","Severity"]].rename(columns={"Duration":"D_SDI","Severity":"S_SDI"})

        merged = pd.merge_asof(
            spi.sort_values("Start"),
            sdi.sort_values("Start"),
            on="Start",
            tolerance=pd.Timedelta("90D"),
            direction="nearest"
        ).dropna()

        if len(merged) >= 5:
            tau = merged["D_SPI"].corr(merged["D_SDI"], method="kendall")
            print(f"{station} SPI-SDI {scale}: {len(merged)} matched  tau_D={tau:.2f}")
        else:
            print(f"{station} SPI-SDI {scale}: {len(merged)} matched  [INSUFFICIENT]")

In [ ]:
"""
Response Probability Contour Plots (Figure 9-10) and
Trigger Threshold Analysis (Figure 13-14)

UPDATED: Date-based event matching (merge_asof) + SPI-RDI primary pair

Requirements:
    pip install pandas numpy matplotlib scipy openpyxl seaborn

Output:
    Response_Probability_v2/ — contour plots
    Trigger_Threshold_v2/    — threshold plots and table
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_RESP = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Response_Probability_v2"
OUTPUT_TRIG = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Trigger_Threshold_v2"

THRESHOLD   = -0.5
STATIONS    = ["Kastamonu", "Sivas", "Kayseri", "Yozgat", "Kirikkale"]
MIN_EVENTS  = 8
MATCH_TOL   = pd.Timedelta("90D")   # matching tolerance

# Analysis pairs — primary: SPI-RDI, secondary: SPI-SDI where data permits
PAIRS = [
    ("SPI", "RDI", [3, 6]),    # strong τ all stations
    ("SPI", "SDI", [3, 6]),    # only Kastamonu + Sivas reliable
]
# ============================================================

os.makedirs(OUTPUT_RESP, exist_ok=True)
os.makedirs(OUTPUT_TRIG, exist_ok=True)

df_all = pd.read_csv(EVENTS_FILE, parse_dates=["Start", "End"])
df_all["Peak"] = df_all["Peak"].abs()
df_ev = df_all[df_all["Threshold"] == THRESHOLD].copy()
print(f"Events loaded: {len(df_ev)}\n")


# ===========================================================
# DATE-BASED EVENT MATCHING
# ===========================================================
def match_events(station, idx_x, idx_y, scale):
    """Match SPI events with RDI/SDI events by nearest Start date."""
    sx = df_ev[(df_ev["Station"]==station) & (df_ev["Index"]==idx_x) &
               (df_ev["Scale"]==scale)][["Start","Duration","Severity","Peak"]].copy()
    sy = df_ev[(df_ev["Station"]==station) & (df_ev["Index"]==idx_y) &
               (df_ev["Scale"]==scale)][["Start","Duration","Severity","Peak"]].copy()

    sx = sx.rename(columns={"Duration":"D_X","Severity":"S_X","Peak":"P_X"})
    sy = sy.rename(columns={"Duration":"D_Y","Severity":"S_Y","Peak":"P_Y"})

    merged = pd.merge_asof(
        sx.sort_values("Start"),
        sy.sort_values("Start"),
        on="Start", tolerance=MATCH_TOL, direction="nearest"
    ).dropna()

    return merged


# ===========================================================
# COPULA HELPERS
# ===========================================================
def empirical_cdf(x):
    rank = stats.rankdata(x, method="average")
    return rank / (len(x) + 1)

def fit_best_marginal(x):
    x = np.array(x); x = x[x > 0]
    best_ks, best_dist, best_params = np.inf, None, None
    for dist in [stats.gamma, stats.lognorm, stats.genextreme,
                 stats.weibull_min, stats.expon]:
        try:
            params = dist.fit(x, floc=0)
            ks, _  = stats.kstest(x, dist.cdf, args=params)
            if ks < best_ks:
                best_ks, best_dist, best_params = ks, dist, params
        except Exception:
            pass
    return best_dist, best_params

def marginal_cdf(x_val, dist, params):
    return np.clip(dist.cdf(x_val, *params), 1e-6, 1-1e-6)

# Copula CDFs
def frank_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    return -1/t*np.log(1+(np.exp(-t*u)-1)*(np.exp(-t*v)-1)/np.maximum(np.exp(-t)-1,eps))

def frank_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    et=np.exp(-t); eu=np.exp(-t*u); ev=np.exp(-t*v)
    return np.maximum(np.abs(-t*(et-1)*eu*ev/((et-1+(eu-1)*(ev-1))**2+eps)),eps)

def gumbel_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    A=(-np.log(u))**t+(-np.log(v))**t
    return np.exp(-A**(1/t))

def gumbel_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    lu=-np.log(u); lv=-np.log(v)
    A=lu**t+lv**t; C=np.exp(-A**(1/t))
    return np.maximum(C*(u*v)**(-1)*A**(-2+2/t)*(lu*lv)**(t-1)*(1+(t-1)*A**(-1/t)),eps)

def clayton_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    return np.maximum(u**(-t)+v**(-t)-1,eps)**(-1/t)

def clayton_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    A=np.maximum(u**(-t)+v**(-t)-1,eps)
    return np.maximum((1+t)*(u*v)**(-(1+t))*A**(-(2+1/t)),eps)

def gaussian_ll(u, v, rho):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    x=stats.norm.ppf(u); y=stats.norm.ppf(v); det=1-rho**2
    return np.sum(-0.5*np.log(det)-(rho**2*(x**2+y**2)-2*rho*x*y)/(2*det))

def gaussian_cdf_scalar(u_s, v_s, rho):
    from scipy.stats import multivariate_normal
    u_s=np.clip(u_s,1e-6,1-1e-6); v_s=np.clip(v_s,1e-6,1-1e-6)
    x=stats.norm.ppf(u_s); y=stats.norm.ppf(v_s)
    return multivariate_normal.cdf([x,y],mean=[0,0],cov=[[1,rho],[rho,1]])

def fit_copula(u, v):
    from scipy.stats import kendalltau
    tau,_ = kendalltau(u,v)
    best  = None

    for name, pdf_fn, bounds, theta0 in [
        ("Frank",   frank_pdf,   (-50,50),    4*tau),
        ("Gumbel",  gumbel_pdf,  (1.001,50),  max(1/(1-tau+1e-5),1.01)),
        ("Clayton", clayton_pdf, (0.01,50),   max(2*tau/(1-tau+1e-5),0.01)),
    ]:
        try:
            res = minimize_scalar(
                lambda t: -np.sum(np.log(np.maximum(pdf_fn(u,v,t),1e-300))),
                bounds=bounds, method="bounded"
            )
            ll  = -res.fun
            aic = 2 - 2*ll
            if best is None or aic < best[3]:
                best = (name, res.x, ll, aic)
        except Exception:
            pass

    # Gaussian
    try:
        res  = minimize_scalar(lambda r: -gaussian_ll(u,v,r),
                               bounds=(-0.999,0.999), method="bounded")
        ll   = -res.fun
        aic  = 2 - 2*ll
        if best is None or aic < best[3]:
            best = ("Gaussian", res.x, ll, aic)
    except Exception:
        pass

    return best   # (family, param, ll, aic)

def copula_cdf(u_val, v_val, family, param):
    u_val=np.clip(u_val,1e-6,1-1e-6); v_val=np.clip(v_val,1e-6,1-1e-6)
    if family=="Frank":   return frank_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Gumbel":  return gumbel_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Clayton": return clayton_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Gaussian":return gaussian_cdf_scalar(u_val, v_val, param)
    return u_val * v_val

def conditional_prob_grid(u_grid, v_grid, family, param):
    """P(V<=v | U=u) via numerical derivative dC/du."""
    h  = 1e-4
    prob = np.zeros((len(v_grid), len(u_grid)))
    for j, uv in enumerate(u_grid):
        u1 = np.clip(uv+h, 1e-6, 1-1e-6)
        u2 = np.clip(uv-h, 1e-6, 1-1e-6)
        for i, vv in enumerate(v_grid):
            c1 = copula_cdf(u1, vv, family, param)
            c2 = copula_cdf(u2, vv, family, param)
            prob[i,j] = np.clip((c1-c2)/(u1-u2), 0, 1)
    return prob


# ===========================================================
# FIGURE 9-10: Response Probability Contour
# ===========================================================
def plot_response_probability(idx_x, idx_y, variable, scale):
    matched_data = {}
    for station in STATIONS:
        m = match_events(station, idx_x, idx_y, scale)
        if len(m) >= MIN_EVENTS:
            matched_data[station] = m

    if not matched_data:
        print(f"  No sufficient data for {idx_x}-{idx_y} {variable} Scale-{scale}")
        return

    n_st  = len(matched_data)
    fig, axes = plt.subplots(1, n_st, figsize=(4*n_st, 5), sharey=True)
    if n_st == 1: axes = [axes]
    fig.patch.set_facecolor("white")

    col_x = "D_X" if variable=="Duration" else "S_X"
    col_y = "D_Y" if variable=="Duration" else "S_Y"
    levels = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
    colors_l = ["#4575b4","#74add1","#abd9e9","#e0f3f8",
                "#fee090","#fdae61","#f46d43","#d73027","#a50026"]

    for ax, (station, m) in zip(axes, matched_data.items()):
        x_raw = m[col_x].values.astype(float)
        y_raw = m[col_y].values.astype(float)

        dist_x, par_x = fit_best_marginal(x_raw)
        dist_y, par_y = fit_best_marginal(y_raw)
        if dist_x is None or dist_y is None:
            ax.set_title(f"{station}\nFit failed"); continue

        u = empirical_cdf(x_raw)
        v = empirical_cdf(y_raw)
        cop = fit_copula(u, v)
        if cop is None:
            ax.set_title(f"{station}\nCopula failed"); continue

        family, param, _, aic = cop
        tau,_ = stats.kendalltau(x_raw, y_raw)

        x_max = max(x_raw.max()*1.1, 5)
        y_max = max(y_raw.max()*1.1, 5)
        x_grid = np.linspace(0.1, x_max, 25)
        y_grid = np.linspace(0.1, y_max, 25)

        u_grid = np.array([marginal_cdf(xv, dist_x, par_x) for xv in x_grid])
        v_grid = np.array([marginal_cdf(yv, dist_y, par_y) for yv in y_grid])

        prob = conditional_prob_grid(u_grid, v_grid, family, param)

        cs = ax.contour(x_grid, y_grid, prob, levels=levels,
                        colors=colors_l, linewidths=1.5)
        ax.clabel(cs, fmt="%.1f", fontsize=7, inline=True)
        ax.scatter(x_raw, y_raw, s=12, color="black", alpha=0.4, zorder=5)
        ax.set_xlim(0, x_max); ax.set_ylim(0, y_max)
        ax.set_xlabel(f"{idx_x}-{scale} {variable}", fontsize=9)
        ax.set_title(f"{station}\n{family} (τ={tau:.2f}, n={len(m)})",
                     fontsize=9, fontweight="bold")
        ax.grid(linestyle="--", alpha=0.3)

    axes[0].set_ylabel(f"{idx_y}-{scale} {variable}", fontsize=9)
    fig.suptitle(
        f"P({idx_y}-{scale} {variable} ≤ y  |  {idx_x}-{scale} {variable} = x)\n"
        f"Date-matched events  |  Threshold = {THRESHOLD}",
        fontsize=11, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_RESP,
          f"ResponseProb_{idx_x}-{idx_y}_{variable}_Scale{scale}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ===========================================================
# FIGURE 13-14: Trigger Threshold
# P(Y<=y_thr | x1<X<=x2) iterating over SPI severity
# ===========================================================
def compute_trigger(station, idx_x, idx_y, scale, y_threshold_pct=50):
    m = match_events(station, idx_x, idx_y, scale)
    if len(m) < MIN_EVENTS:
        return None, None

    x_raw = m["S_X"].values.astype(float)
    y_raw = m["S_Y"].values.astype(float)

    dist_x, par_x = fit_best_marginal(x_raw)
    dist_y, par_y = fit_best_marginal(y_raw)
    if dist_x is None or dist_y is None:
        return None, None

    u = empirical_cdf(x_raw)
    v = empirical_cdf(y_raw)
    cop = fit_copula(u, v)
    if cop is None:
        return None, None

    family, param = cop[0], cop[1]

    # Y threshold: median severity of Y series
    y_thr = np.percentile(y_raw, y_threshold_pct)
    v_thr = marginal_cdf(y_thr, dist_y, par_y)

    # Iterate X (SPI severity) from min to max in steps
    x_vals = np.arange(0.5, x_raw.max()*1.05, 0.5)
    probs  = []
    x_out  = []

    for xi in x_vals:
        xi0 = max(xi - 0.5, 0.01)
        u2  = marginal_cdf(xi,  dist_x, par_x)
        u1  = marginal_cdf(xi0, dist_x, par_x)
        denom = u2 - u1
        if denom < 1e-8: continue

        c2 = copula_cdf(u2, v_thr, family, param)
        c1 = copula_cdf(u1, v_thr, family, param)
        p  = np.clip((c2-c1)/denom, 0, 1)
        probs.append(p)
        x_out.append(xi)

    return np.array(x_out), np.array(probs)


def plot_trigger_threshold(idx_x, idx_y, scale):
    colors = {"Kastamonu":"#1f77b4","Sivas":"#ff7f0e",
              "Kayseri":"#2ca02c","Yozgat":"#d62728","Kirikkale":"#9467bd"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("white")

    records = []

    # Panel 1: Trigger curves
    ax = axes[0]
    for station in STATIONS:
        xv, pv = compute_trigger(station, idx_x, idx_y, scale)
        if xv is None: continue
        ax.plot(xv, pv, "-o", markersize=4, linewidth=1.8,
                color=colors[station], label=station)
        # Find 50% threshold
        if pv.max() >= 0.5:
            idx50 = np.argmin(np.abs(pv - 0.5))
            ax.axvline(xv[idx50], color=colors[station],
                       linestyle=":", linewidth=0.8, alpha=0.6)
            records.append({"Station":station,"Pair":f"{idx_x}-{idx_y}",
                            "Scale":scale,"Thr_50pct":round(xv[idx50],2),
                            "Max_prob":round(pv.max(),3)})

    ax.axhline(0.5, color="black", linewidth=1.5, linestyle="--",
               label="50% level")
    ax.set_xlabel(f"{idx_x}-{scale} Severity (trigger variable)", fontsize=10)
    ax.set_ylabel(f"P({idx_y} Severity ≥ median | {idx_x} Severity in interval)",
                  fontsize=9)
    ax.set_ylim(0, 1); ax.set_xlim(left=0)
    ax.set_title(f"Trigger Probability: {idx_x}-{scale} → {idx_y}-{scale}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(linestyle="--", alpha=0.4)

    # Panel 2: Grade propagation — 4 SPI grades x 4 SDI levels
    ax2 = axes[1]
    grade_bounds = {"Mild":(0.5,2),"Moderate":(2,4),"Severe":(4,7),"Extreme":(7,30)}
    grade_levels = {"Mild":1.5,"Moderate":3.5,"Severe":6,"Extreme":12}
    colors_g = {"Mild":"#74add1","Moderate":"#fdae61",
                "Severe":"#f46d43","Extreme":"#a50026"}
    markers  = {"Mild":"o","Moderate":"s","Severe":"^","Extreme":"D"}

    plotted = False
    for station in STATIONS:
        m = match_events(station, idx_x, idx_y, scale)
        if len(m) < MIN_EVENTS: continue

        x_raw = m["S_X"].values.astype(float)
        y_raw = m["S_Y"].values.astype(float)
        dist_x, par_x = fit_best_marginal(x_raw)
        dist_y, par_y = fit_best_marginal(y_raw)
        if dist_x is None: continue
        u = empirical_cdf(x_raw); v = empirical_cdf(y_raw)
        cop = fit_copula(u, v)
        if cop is None: continue
        family, param = cop[0], cop[1]

        # Use first valid station for grade plot
        for spi_grade, (lo, hi) in grade_bounds.items():
            u2 = marginal_cdf(hi, dist_x, par_x)
            u1 = marginal_cdf(lo, dist_x, par_x)
            denom = u2 - u1
            if denom < 1e-8: continue
            probs_g = []
            for sdi_grade, sdi_thr in grade_levels.items():
                v_t = marginal_cdf(sdi_thr, dist_y, par_y)
                c2  = copula_cdf(u2, v_t, family, param)
                c1  = copula_cdf(u1, v_t, family, param)
                probs_g.append(np.clip((c2-c1)/denom, 0, 1))
            ax2.plot(list(grade_levels.keys()), probs_g,
                     "-"+markers[spi_grade], color=colors_g[spi_grade],
                     label=f"SPI {spi_grade}", linewidth=1.8, markersize=7)
        ax2.set_title(f"Grade Propagation — {station}\n"
                      f"{idx_x}-{scale} → {idx_y}-{scale}",
                      fontsize=10, fontweight="bold")
        plotted = True
        break   # show one representative station

    if not plotted:
        ax2.text(0.5,0.5,"Insufficient data",ha="center",va="center",
                 transform=ax2.transAxes)

    ax2.set_xlabel(f"{idx_y} Drought Grade", fontsize=10)
    ax2.set_ylabel("Conditional Probability", fontsize=10)
    ax2.set_ylim(0, 1)
    ax2.legend(fontsize=9, title="SPI Drought Grade")
    ax2.grid(linestyle="--", alpha=0.4)

    fig.suptitle(
        f"Trigger Thresholds & Grade Propagation — {idx_x}-{scale} → {idx_y}-{scale}\n"
        f"Date-matched events  |  Threshold = {THRESHOLD}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_TRIG,
          f"TriggerThreshold_{idx_x}-{idx_y}_Scale{scale}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")
    return records


# ===========================================================
# MAIN
# ===========================================================
print("="*60)
print("  Response Probability & Trigger Threshold  (v2)")
print("="*60+"\n")

all_records = []

for idx_x, idx_y, scales in PAIRS:
    for scale in scales:
        print(f"\n--- {idx_x} → {idx_y}  Scale-{scale} ---")
        for var in ["Duration", "Severity"]:
            print(f"  [{var}] Response probability...")
            plot_response_probability(idx_x, idx_y, var, scale)
        print(f"  [Trigger] Threshold analysis...")
        recs = plot_trigger_threshold(idx_x, idx_y, scale)
        if recs: all_records.extend(recs)

# Save trigger threshold table
if all_records:
    df_trig = pd.DataFrame(all_records)
    df_trig.to_excel(os.path.join(OUTPUT_TRIG,"trigger_threshold_table_v2.xlsx"), index=False)
    df_trig.to_csv( os.path.join(OUTPUT_TRIG,"trigger_threshold_table_v2.csv"),  index=False)
    print("\nTrigger threshold table:")
    print(df_trig.to_string(index=False))

print(f"\nAll done!")
print(f"  Response Prob  -> {OUTPUT_RESP}")
print(f"  Trigger Thresh -> {OUTPUT_TRIG}")


In [ ]:
"""
Response Probability Contour Plots (Figure 9-10) and
Trigger Threshold Analysis (Figure 13-14)

UPDATED: Date-based event matching (merge_asof) + SPI-RDI primary pair

Requirements:
    pip install pandas numpy matplotlib scipy openpyxl seaborn

Output:
    Response_Probability_v2/ — contour plots
    Trigger_Threshold_v2/    — threshold plots and table
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_RESP = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Response_Probability_v2"
OUTPUT_TRIG = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Trigger_Threshold_v2"

THRESHOLD   = -0.5
STATIONS    = ["Kastamonu", "Sivas", "Kayseri", "Yozgat", "Kirikkale"]
MIN_EVENTS  = 8
MATCH_TOL   = pd.Timedelta("90D")   # matching tolerance

# Analysis pairs — primary: SPI-RDI, secondary: SPI-SDI where data permits
PAIRS = [
    ("SPI", "RDI", [3, 6]),    # strong τ all stations
    ("SPI", "SDI", [3, 6]),    # only Kastamonu + Sivas reliable
]
# ============================================================

os.makedirs(OUTPUT_RESP, exist_ok=True)
os.makedirs(OUTPUT_TRIG, exist_ok=True)

df_all = pd.read_csv(EVENTS_FILE, parse_dates=["Start", "End"])
df_all["Peak"] = df_all["Peak"].abs()
df_ev = df_all[df_all["Threshold"] == THRESHOLD].copy()
print(f"Events loaded: {len(df_ev)}\n")


# ===========================================================
# DATE-BASED EVENT MATCHING
# ===========================================================
def match_events(station, idx_x, idx_y, scale):
    """Match SPI events with RDI/SDI events by nearest Start date."""
    sx = df_ev[(df_ev["Station"]==station) & (df_ev["Index"]==idx_x) &
               (df_ev["Scale"]==scale)][["Start","Duration","Severity","Peak"]].copy()
    sy = df_ev[(df_ev["Station"]==station) & (df_ev["Index"]==idx_y) &
               (df_ev["Scale"]==scale)][["Start","Duration","Severity","Peak"]].copy()

    sx = sx.rename(columns={"Duration":"D_X","Severity":"S_X","Peak":"P_X"})
    sy = sy.rename(columns={"Duration":"D_Y","Severity":"S_Y","Peak":"P_Y"})

    merged = pd.merge_asof(
        sx.sort_values("Start"),
        sy.sort_values("Start"),
        on="Start", tolerance=MATCH_TOL, direction="nearest"
    ).dropna()

    return merged


# ===========================================================
# COPULA HELPERS
# ===========================================================
def empirical_cdf(x):
    rank = stats.rankdata(x, method="average")
    return rank / (len(x) + 1)

def fit_best_marginal(x):
    x = np.array(x); x = x[x > 0]
    best_ks, best_dist, best_params = np.inf, None, None
    for dist in [stats.gamma, stats.lognorm, stats.genextreme,
                 stats.weibull_min, stats.expon]:
        try:
            params = dist.fit(x, floc=0)
            ks, _  = stats.kstest(x, dist.cdf, args=params)
            if ks < best_ks:
                best_ks, best_dist, best_params = ks, dist, params
        except Exception:
            pass
    return best_dist, best_params

def marginal_cdf(x_val, dist, params):
    return np.clip(dist.cdf(x_val, *params), 1e-6, 1-1e-6)

# Copula CDFs
def frank_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    return -1/t*np.log(1+(np.exp(-t*u)-1)*(np.exp(-t*v)-1)/np.maximum(np.exp(-t)-1,eps))

def frank_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    et=np.exp(-t); eu=np.exp(-t*u); ev=np.exp(-t*v)
    return np.maximum(np.abs(-t*(et-1)*eu*ev/((et-1+(eu-1)*(ev-1))**2+eps)),eps)

def gumbel_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    A=(-np.log(u))**t+(-np.log(v))**t
    return np.exp(-A**(1/t))

def gumbel_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    lu=-np.log(u); lv=-np.log(v)
    A=lu**t+lv**t; C=np.exp(-A**(1/t))
    return np.maximum(C*(u*v)**(-1)*A**(-2+2/t)*(lu*lv)**(t-1)*(1+(t-1)*A**(-1/t)),eps)

def clayton_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    return np.maximum(u**(-t)+v**(-t)-1,eps)**(-1/t)

def clayton_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    A=np.maximum(u**(-t)+v**(-t)-1,eps)
    return np.maximum((1+t)*(u*v)**(-(1+t))*A**(-(2+1/t)),eps)

def gaussian_ll(u, v, rho):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    x=stats.norm.ppf(u); y=stats.norm.ppf(v); det=1-rho**2
    return np.sum(-0.5*np.log(det)-(rho**2*(x**2+y**2)-2*rho*x*y)/(2*det))

def gaussian_cdf_scalar(u_s, v_s, rho):
    from scipy.stats import multivariate_normal
    u_s=np.clip(u_s,1e-6,1-1e-6); v_s=np.clip(v_s,1e-6,1-1e-6)
    x=stats.norm.ppf(u_s); y=stats.norm.ppf(v_s)
    return multivariate_normal.cdf([x,y],mean=[0,0],cov=[[1,rho],[rho,1]])

def fit_copula(u, v):
    from scipy.stats import kendalltau
    tau,_ = kendalltau(u,v)
    best  = None

    for name, pdf_fn, bounds, theta0 in [
        ("Frank",   frank_pdf,   (-50,50),    4*tau),
        ("Gumbel",  gumbel_pdf,  (1.001,50),  max(1/(1-tau+1e-5),1.01)),
        ("Clayton", clayton_pdf, (0.01,50),   max(2*tau/(1-tau+1e-5),0.01)),
    ]:
        try:
            res = minimize_scalar(
                lambda t: -np.sum(np.log(np.maximum(pdf_fn(u,v,t),1e-300))),
                bounds=bounds, method="bounded"
            )
            ll  = -res.fun
            aic = 2 - 2*ll
            if best is None or aic < best[3]:
                best = (name, res.x, ll, aic)
        except Exception:
            pass

    # Gaussian
    try:
        res  = minimize_scalar(lambda r: -gaussian_ll(u,v,r),
                               bounds=(-0.999,0.999), method="bounded")
        ll   = -res.fun
        aic  = 2 - 2*ll
        if best is None or aic < best[3]:
            best = ("Gaussian", res.x, ll, aic)
    except Exception:
        pass

    return best   # (family, param, ll, aic)

def copula_cdf(u_val, v_val, family, param):
    u_val=np.clip(u_val,1e-6,1-1e-6); v_val=np.clip(v_val,1e-6,1-1e-6)
    if family=="Frank":   return frank_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Gumbel":  return gumbel_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Clayton": return clayton_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Gaussian":return gaussian_cdf_scalar(u_val, v_val, param)
    return u_val * v_val

def conditional_prob_grid(u_grid, v_grid, family, param):
    """P(V<=v | U=u) via numerical derivative dC/du."""
    h  = 1e-4
    prob = np.zeros((len(v_grid), len(u_grid)))
    for j, uv in enumerate(u_grid):
        u1 = np.clip(uv+h, 1e-6, 1-1e-6)
        u2 = np.clip(uv-h, 1e-6, 1-1e-6)
        for i, vv in enumerate(v_grid):
            c1 = copula_cdf(u1, vv, family, param)
            c2 = copula_cdf(u2, vv, family, param)
            prob[i,j] = np.clip((c1-c2)/(u1-u2), 0, 1)
    return prob


# ===========================================================
# FIGURE 9-10: Response Probability Contour
# ===========================================================
def plot_response_probability(idx_x, idx_y, variable, scale):
    matched_data = {}
    for station in STATIONS:
        m = match_events(station, idx_x, idx_y, scale)
        if len(m) >= MIN_EVENTS:
            matched_data[station] = m

    if not matched_data:
        print(f"  No sufficient data for {idx_x}-{idx_y} {variable} Scale-{scale}")
        return

    n_st  = len(matched_data)
    fig, axes = plt.subplots(1, n_st, figsize=(4*n_st, 5), sharey=True)
    if n_st == 1: axes = [axes]
    fig.patch.set_facecolor("white")

    col_x = "D_X" if variable=="Duration" else "S_X"
    col_y = "D_Y" if variable=="Duration" else "S_Y"
    levels = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
    colors_l = ["#4575b4","#74add1","#abd9e9","#e0f3f8",
                "#fee090","#fdae61","#f46d43","#d73027","#a50026"]

    for ax, (station, m) in zip(axes, matched_data.items()):
        x_raw = m[col_x].values.astype(float)
        y_raw = m[col_y].values.astype(float)

        dist_x, par_x = fit_best_marginal(x_raw)
        dist_y, par_y = fit_best_marginal(y_raw)
        if dist_x is None or dist_y is None:
            ax.set_title(f"{station}\nFit failed"); continue

        u = empirical_cdf(x_raw)
        v = empirical_cdf(y_raw)
        cop = fit_copula(u, v)
        if cop is None:
            ax.set_title(f"{station}\nCopula failed"); continue

        family, param, _, aic = cop

        # Frank high-theta numerical fix — replace with next best copula
        if family == "Frank" and abs(param) > 8:
            candidates = []
            for name, pdf_fn, bounds in [
                ("Gumbel",  gumbel_pdf,  (1.001, 50)),
                ("Clayton", clayton_pdf, (0.01,  50)),
            ]:
                try:
                    res = minimize_scalar(
                        lambda t, fn=pdf_fn: -np.sum(np.log(np.maximum(fn(u,v,t),1e-300))),
                        bounds=bounds, method="bounded"
                    )
                    candidates.append((name, res.x, -res.fun, 2+2*res.fun))
                except Exception:
                    pass
            try:
                res2 = minimize_scalar(lambda r: -gaussian_ll(u,v,r),
                                       bounds=(-0.999,0.999), method="bounded")
                candidates.append(("Gaussian", res2.x, -res2.fun, 2+2*res2.fun))
            except Exception:
                pass
            if candidates:
                candidates.sort(key=lambda x: x[3])
                family, param = candidates[0][0], candidates[0][1]
                print(f"    [FIX] {station}: Frank replaced by {family} (theta={param:.3f})")

        tau,_ = stats.kendalltau(x_raw, y_raw)

        x_max = max(x_raw.max()*1.1, 5)
        y_max = max(y_raw.max()*1.1, 5)
        x_grid = np.linspace(0.1, x_max, 25)
        y_grid = np.linspace(0.1, y_max, 25)

        u_grid = np.array([marginal_cdf(xv, dist_x, par_x) for xv in x_grid])
        v_grid = np.array([marginal_cdf(yv, dist_y, par_y) for yv in y_grid])

        prob = conditional_prob_grid(u_grid, v_grid, family, param)

        cs = ax.contour(x_grid, y_grid, prob, levels=levels,
                        colors=colors_l, linewidths=1.5)
        ax.clabel(cs, fmt="%.1f", fontsize=7, inline=True)
        ax.scatter(x_raw, y_raw, s=12, color="black", alpha=0.4, zorder=5)
        ax.set_xlim(0, x_max); ax.set_ylim(0, y_max)
        ax.set_xlabel(f"{idx_x}-{scale} {variable}", fontsize=9)
        ax.set_title(f"{station}\n{family} (τ={tau:.2f}, n={len(m)})",
                     fontsize=9, fontweight="bold")
        ax.grid(linestyle="--", alpha=0.3)

    axes[0].set_ylabel(f"{idx_y}-{scale} {variable}", fontsize=9)
    fig.suptitle(
        f"P({idx_y}-{scale} {variable} ≤ y  |  {idx_x}-{scale} {variable} = x)\n"
        f"Date-matched events  |  Threshold = {THRESHOLD}",
        fontsize=11, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_RESP,
          f"ResponseProb_{idx_x}-{idx_y}_{variable}_Scale{scale}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ===========================================================
# FIGURE 13-14: Trigger Threshold
# P(Y<=y_thr | x1<X<=x2) iterating over SPI severity
# ===========================================================
def compute_trigger(station, idx_x, idx_y, scale, y_threshold_pct=50):
    m = match_events(station, idx_x, idx_y, scale)
    if len(m) < MIN_EVENTS:
        return None, None

    x_raw = m["S_X"].values.astype(float)
    y_raw = m["S_Y"].values.astype(float)

    dist_x, par_x = fit_best_marginal(x_raw)
    dist_y, par_y = fit_best_marginal(y_raw)
    if dist_x is None or dist_y is None:
        return None, None

    u = empirical_cdf(x_raw)
    v = empirical_cdf(y_raw)
    cop = fit_copula(u, v)
    if cop is None:
        return None, None

    family, param = cop[0], cop[1]

    # Y threshold: median severity of Y series
    y_thr = np.percentile(y_raw, y_threshold_pct)
    v_thr = marginal_cdf(y_thr, dist_y, par_y)

    # Iterate X (SPI severity) from min to max in steps
    x_vals = np.arange(0.5, x_raw.max()*1.05, 0.5)
    probs  = []
    x_out  = []

    for xi in x_vals:
        xi0 = max(xi - 0.5, 0.01)
        u2  = marginal_cdf(xi,  dist_x, par_x)
        u1  = marginal_cdf(xi0, dist_x, par_x)
        denom = u2 - u1
        if denom < 1e-8: continue

        c2 = copula_cdf(u2, v_thr, family, param)
        c1 = copula_cdf(u1, v_thr, family, param)
        p  = np.clip((c2-c1)/denom, 0, 1)
        probs.append(p)
        x_out.append(xi)

    return np.array(x_out), np.array(probs)


def plot_trigger_threshold(idx_x, idx_y, scale):
    colors = {"Kastamonu":"#1f77b4","Sivas":"#ff7f0e",
              "Kayseri":"#2ca02c","Yozgat":"#d62728","Kirikkale":"#9467bd"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("white")

    records = []

    # Panel 1: Trigger curves
    ax = axes[0]
    for station in STATIONS:
        xv, pv = compute_trigger(station, idx_x, idx_y, scale)
        if xv is None: continue
        ax.plot(xv, pv, "-o", markersize=4, linewidth=1.8,
                color=colors[station], label=station)
        # Find 50% threshold
        if pv.max() >= 0.5:
            idx50 = np.argmin(np.abs(pv - 0.5))
            ax.axvline(xv[idx50], color=colors[station],
                       linestyle=":", linewidth=0.8, alpha=0.6)
            records.append({"Station":station,"Pair":f"{idx_x}-{idx_y}",
                            "Scale":scale,"Thr_50pct":round(xv[idx50],2),
                            "Max_prob":round(pv.max(),3)})

    ax.axhline(0.5, color="black", linewidth=1.5, linestyle="--",
               label="50% level")
    ax.set_xlabel(f"{idx_x}-{scale} Severity (trigger variable)", fontsize=10)
    ax.set_ylabel(f"P({idx_y} Severity ≥ median | {idx_x} Severity in interval)",
                  fontsize=9)
    ax.set_ylim(0, 1); ax.set_xlim(left=0)
    ax.set_title(f"Trigger Probability: {idx_x}-{scale} → {idx_y}-{scale}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(linestyle="--", alpha=0.4)

    # Panel 2: Grade propagation — 4 SPI grades x 4 SDI levels
    ax2 = axes[1]
    grade_bounds = {"Mild":(0.5,2),"Moderate":(2,4),"Severe":(4,7),"Extreme":(7,30)}
    grade_levels = {"Mild":1.5,"Moderate":3.5,"Severe":6,"Extreme":12}
    colors_g = {"Mild":"#74add1","Moderate":"#fdae61",
                "Severe":"#f46d43","Extreme":"#a50026"}
    markers  = {"Mild":"o","Moderate":"s","Severe":"^","Extreme":"D"}

    plotted = False
    for station in STATIONS:
        m = match_events(station, idx_x, idx_y, scale)
        if len(m) < MIN_EVENTS: continue

        x_raw = m["S_X"].values.astype(float)
        y_raw = m["S_Y"].values.astype(float)
        dist_x, par_x = fit_best_marginal(x_raw)
        dist_y, par_y = fit_best_marginal(y_raw)
        if dist_x is None: continue
        u = empirical_cdf(x_raw); v = empirical_cdf(y_raw)
        cop = fit_copula(u, v)
        if cop is None: continue
        family, param = cop[0], cop[1]

        # Use first valid station for grade plot
        for spi_grade, (lo, hi) in grade_bounds.items():
            u2 = marginal_cdf(hi, dist_x, par_x)
            u1 = marginal_cdf(lo, dist_x, par_x)
            denom = u2 - u1
            if denom < 1e-8: continue
            probs_g = []
            for sdi_grade, sdi_thr in grade_levels.items():
                v_t = marginal_cdf(sdi_thr, dist_y, par_y)
                c2  = copula_cdf(u2, v_t, family, param)
                c1  = copula_cdf(u1, v_t, family, param)
                probs_g.append(np.clip((c2-c1)/denom, 0, 1))
            ax2.plot(list(grade_levels.keys()), probs_g,
                     "-"+markers[spi_grade], color=colors_g[spi_grade],
                     label=f"SPI {spi_grade}", linewidth=1.8, markersize=7)
        ax2.set_title(f"Grade Propagation — {station}\n"
                      f"{idx_x}-{scale} → {idx_y}-{scale}",
                      fontsize=10, fontweight="bold")
        plotted = True
        break   # show one representative station

    if not plotted:
        ax2.text(0.5,0.5,"Insufficient data",ha="center",va="center",
                 transform=ax2.transAxes)

    ax2.set_xlabel(f"{idx_y} Drought Grade", fontsize=10)
    ax2.set_ylabel("Conditional Probability", fontsize=10)
    ax2.set_ylim(0, 1)
    ax2.legend(fontsize=9, title="SPI Drought Grade")
    ax2.grid(linestyle="--", alpha=0.4)

    fig.suptitle(
        f"Trigger Thresholds & Grade Propagation — {idx_x}-{scale} → {idx_y}-{scale}\n"
        f"Date-matched events  |  Threshold = {THRESHOLD}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_TRIG,
          f"TriggerThreshold_{idx_x}-{idx_y}_Scale{scale}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")
    return records


# ===========================================================
# MAIN
# ===========================================================
print("="*60)
print("  Response Probability & Trigger Threshold  (v2)")
print("="*60+"\n")

all_records = []

for idx_x, idx_y, scales in PAIRS:
    for scale in scales:
        print(f"\n--- {idx_x} → {idx_y}  Scale-{scale} ---")
        for var in ["Duration", "Severity"]:
            print(f"  [{var}] Response probability...")
            plot_response_probability(idx_x, idx_y, var, scale)
        print(f"  [Trigger] Threshold analysis...")
        recs = plot_trigger_threshold(idx_x, idx_y, scale)
        if recs: all_records.extend(recs)

# Save trigger threshold table
if all_records:
    df_trig = pd.DataFrame(all_records)
    df_trig.to_excel(os.path.join(OUTPUT_TRIG,"trigger_threshold_table_v2.xlsx"), index=False)
    df_trig.to_csv( os.path.join(OUTPUT_TRIG,"trigger_threshold_table_v2.csv"),  index=False)
    print("\nTrigger threshold table:")
    print(df_trig.to_string(index=False))

print(f"\nAll done!")
print(f"  Response Prob  -> {OUTPUT_RESP}")
print(f"  Trigger Thresh -> {OUTPUT_TRIG}")

In [ ]:
pip install pandas scipy

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# 1. Veriyi Yükle
df = pd.read_csv('/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv')

# Test edilecek dağılımlar
# genextreme: GEV, gamma: Gamma, lognorm: Lognormal,
# weibull_min: Weibull, fisk: Log-Logistic, expon: Üstel
distributions = {
    "GEV": stats.genextreme,
    "Gamma": stats.gamma,
    "Lognormal": stats.lognorm,
    "Weibull": stats.weibull_min,
    "Log-Logistic": stats.fisk,
    "Exponential": stats.expon
}

def fit_best_distribution(data):
    results = []
    for name, dist in distributions.items():
        try:
            # Parametre tahmini (MLE)
            params = dist.fit(data)

            # Kolmogorov-Smirnov Testi
            D, p_value = stats.kstest(data, name if name != "GEV" else "genextreme", args=params)

            results.append({
                "Distribution": name,
                "Params": params,
                "KS_Stat": round(D, 4),
                "P_Value": round(p_value, 4)
            })
        except:
            continue

    # En düşük KS istatistiğine göre sırala (En iyi uyum)
    results_df = pd.DataFrame(results).sort_values(by="KS_Stat")
    return results_df.iloc[0]

# 2. Analizi Gruplandırarak Çalıştır (Her istasyon ve indeks için)
final_results = []

# İstasyonlara ve İndekslere göre grupla (Örn: Kastamonu - RDI)
groups = df.groupby(['Station', 'Index'])

for (station, index_type), group in groups:
    # Hem Duration hem Intensity için ayrı ayrı fit et
    for var_name in ['Duration', 'Intensity']:
        data = group[var_name].values
        best_fit = fit_best_distribution(data)

        final_results.append({
            "Bölge": station,
            "İndeks": index_type,
            "Değişken": var_name,
            "En İyi Dağılım": best_fit['Distribution'],
            "Parametreler": best_fit['Params'],
            "KS Değeri": best_fit['KS_Stat']
        })

# 3. Sonuçları Tablo Formatında Göster
results_table = pd.DataFrame(final_results)
print(results_table.to_string())

# Opsiyonel: Excel/CSV olarak kaydet
#results_table.to_csv("/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/marginal_distribution_results.csv", index=False)
import os
import pandas as pd

# 1. Drive yolunu ve klasörü garantiye alalım
drive_yolu = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari"
if not os.path.exists(drive_yolu):
    os.makedirs(drive_yolu)

# 2. Hangi değişkenin dolu olduğunu kontrol edip kaydetme işlemini yapalım
# (Yukarıda hesapladığınız tablo hangi isimdeyse onu bulup 'df_to_save' değişkenine atar)
df_to_save = None

if 'results_table' in locals():
    df_to_save = results_table
elif 'final_df' in locals():
    df_to_save = final_df
elif 'results_df' in locals():
    df_to_save = results_df

# 3. Kayıt İşlemi
if df_to_save is not None:
    tam_yol = os.path.join(drive_yolu, "marginal_distribution_results.csv")
    df_to_save.to_csv(tam_yol, index=False, encoding='utf-8-sig')
    print("-" * 50)
    print(f"BAŞARILI: Tablo Drive'a şu isimle kaydedildi:\n{tam_yol}")
    print("-" * 50)
else:
    print("HATA: Kaydedilecek tablo (DataFrame) bulunamadı. Lütfen hesaplama yaptığınız hücreyi tekrar çalıştırın.")

In [ ]:
 pip install pandas numpy scipy pymannkendall openpyxl xlsxwriter

In [ ]:
"""
Table 4 — Sequential Mann-Kendall Trend Analysis Summary
Computes Z, p-value, Sen's slope, trend direction, and change point year
for all station × index × scale combinations.

Requirements:
    pip install pandas numpy scipy pymannkendall openpyxl xlsxwriter
"""

import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy.stats import kendalltau
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
INPUT_PATH = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari"

STATION_FILES = {
    "Kastamonu" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kastamonu_output_indices",
    "Sivas"     : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Sivas_output_indices",
    "Kayseri"   : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kayseri_output_indices",
    "Yozgat"    : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_output_indices",
    "Kirikkale" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kirikkale_output_indices",
}

DATE_COL   = "Date"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Tables"
INDEX_COLS = ["SPI-3","SPI-6","SPI-12","SDI-3","SDI-6","SDI-12","RDI-3","RDI-6","RDI-12"]
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ----------------------------------------------------------
# Load station data
# ----------------------------------------------------------
def load_station(path, date_col):
    if os.path.isdir(path):
        files = sorted([
            os.path.join(path, f) for f in os.listdir(path)
            if f.endswith((".xlsx",".xls",".csv")) and not f.startswith("~")
        ])
        frames = []
        seen   = set()
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            tmp = pd.read_excel(fp, parse_dates=[date_col]) if ext in [".xlsx",".xls"] \
                  else pd.read_csv(fp, parse_dates=[date_col])
            new_cols = [date_col] + [c for c in tmp.columns
                                     if c != date_col and c not in seen]
            tmp = tmp[new_cols]
            seen.update(tmp.columns)
            frames.append(tmp)
        df = frames[0]
        for f in frames[1:]:
            df = pd.merge(df, f, on=date_col, how="outer")
    else:
        ext = os.path.splitext(path)[1].lower()
        df  = pd.read_excel(path, parse_dates=[date_col]) if ext in [".xlsx",".xls"] \
              else pd.read_csv(path, parse_dates=[date_col])
    return df.sort_values(date_col).reset_index(drop=True)


# ----------------------------------------------------------
# Change point detection from SQMK (UF/UB intersection)
# ----------------------------------------------------------
def sequential_mk_changepoint(x):
    """Returns change point year index (None if not found)."""
    n  = len(x)
    uf = np.zeros(n)
    for i in range(1, n):
        s     = sum(np.sign(x[i] - x[j]) for j in range(i))
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        uf[i] = (s-e_s)/np.sqrt(var_s) if var_s > 0 else 0.0

    x_rev  = x[::-1]
    uf_rev = np.zeros(n)
    for i in range(1, n):
        s     = sum(np.sign(x_rev[i] - x_rev[j]) for j in range(i))
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        uf_rev[i] = (s-e_s)/np.sqrt(var_s) if var_s > 0 else 0.0
    ub = -uf_rev[::-1]

    # Find intersection within ±1.96 band
    for i in range(1, n):
        d_prev = uf[i-1] - ub[i-1]
        d_curr = uf[i]   - ub[i]
        if d_prev * d_curr < 0:
            if abs(uf[i]) < 1.96 and abs(ub[i]) < 1.96:
                return i   # index of change point
    return None


# ----------------------------------------------------------
# Main computation
# ----------------------------------------------------------
print("="*60)
print("  Table 4 — Mann-Kendall Trend Analysis")
print("="*60 + "\n")

records = []

for station, path in STATION_FILES.items():
    print(f">>> {station}")
    try:
        df = load_station(path, DATE_COL)
    except Exception as e:
        print(f"  [ERROR] {e}"); continue

    # Annual means for trend analysis
    df_y = df.set_index(DATE_COL).resample("YE").mean().dropna(how="all")
    years = df_y.index.year.values

    for col in INDEX_COLS:
        if col not in df_y.columns:
            print(f"  [SKIP] {col} not found"); continue

        series = df_y[col].dropna().values
        if len(series) < 5:
            continue

        idx_name = col.split("-")[0]
        scale    = int(col.split("-")[1])

        # pymannkendall
        try:
            res = mk.original_test(series)
            z_stat    = round(res.z,      3)
            p_val     = round(res.p,      4)
            slope     = round(res.slope,  5)
            tau       = round(res.Tau,    3)
            trend_dir = res.trend   # "increasing" / "decreasing" / "no trend"
        except Exception:
            z_stat = p_val = slope = tau = np.nan
            trend_dir = "N/A"

        # Change point year
        cp_idx = sequential_mk_changepoint(series)
        if cp_idx is not None and cp_idx < len(years):
            cp_year = int(years[cp_idx])
        else:
            cp_year = np.nan

        # Significance label
        if pd.isna(p_val):
            sig = "N/A"
        elif p_val < 0.01:
            sig = "**"
        elif p_val < 0.05:
            sig = "*"
        else:
            sig = "ns"

        # Trend symbol
        if trend_dir == "increasing":
            trend_sym = "↑"
        elif trend_dir == "decreasing":
            trend_sym = "↓"
        else:
            trend_sym = "—"

        records.append({
            "Station"         : station,
            "Index"           : idx_name,
            "Scale (month)"   : scale,
            "N (years)"       : len(series),
            "Kendall τ"       : tau,
            "Z statistic"     : z_stat,
            "p-value"         : p_val,
            "Significance"    : sig,
            "Sen's Slope"     : slope,
            "Trend"           : trend_sym,
            "Change Point Year": cp_year,
        })

        print(f"  {col}: Z={z_stat:+.3f}  p={p_val:.4f} {sig}  "
              f"slope={slope:.5f}  trend={trend_sym}  CP={cp_year}")

    print()

df_res = pd.DataFrame(records)
df_res = df_res.sort_values(["Station","Index","Scale (month)"]).reset_index(drop=True)

# ----------------------------------------------------------
# Save CSV
# ----------------------------------------------------------
csv_path = os.path.join(OUTPUT_DIR, "Table4_MK_Trend.csv")
df_res.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}")

# ----------------------------------------------------------
# Publication-ready Excel
# ----------------------------------------------------------
wb  = Workbook()
ws  = wb.active
ws.title = "Table 4 – MK Trend Analysis"

thin  = Side(style="thin",   color="000000")
thick = Side(style="medium", color="000000")
tb    = Border(left=thin, right=thin, top=thin, bottom=thin)

C_H1   = "1F3864"   # dark navy
C_H2   = "2E75B6"   # mid blue
C_ALT  = "EBF3FB"   # light blue
C_SIG1 = "C6EFCE"   # green — significant *
C_SIG2 = "FFEB9C"   # yellow — near significant
C_NS   = "FFFFFF"   # white — not significant
C_INC  = "FFC7CE"   # red tint — increasing trend
C_DEC  = "C6EFCE"   # green tint — decreasing trend

station_colors = {
    "Kastamonu": "E8F4FD", "Kayseri":  "FEF9E7",
    "Kirikkale": "EAFAF1", "Sivas":    "FDEDEC",
    "Yozgat":   "F4ECF7",
}

def hc(ws, r, c, val, bg, fg="FFFFFF", bold=True, align="center",
        wrap=False, sz=10):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(bold=bold, color=fg, size=sz, name="Calibri")
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal=align, vertical="center",
                               wrap_text=wrap)
    cell.border    = tb
    return cell

def dc(ws, r, c, val, bg="FFFFFF", bold=False, align="center", fmt=None):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(bold=bold, color="000000", size=9, name="Calibri")
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal=align, vertical="center")
    cell.border    = tb
    if fmt: cell.number_format = fmt
    return cell

# Title
n_cols = len(df_res.columns)
ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
t = ws.cell(row=1, column=1,
    value="Table 4. Sequential Mann-Kendall Trend Analysis Results — "
          "Kızılırmak Basin (Annual means of monthly drought indices)")
t.font      = Font(bold=True, size=11, name="Calibri")
t.fill      = PatternFill("solid", fgColor="D9E2F3")
t.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
ws.row_dimensions[1].height = 30

# Group headers
groups = [
    ("Station Info",    1,  4,  C_H1),
    ("MK Statistics",   5,  9,  C_H2),
    ("Trend Result",   10, 11,  C_H1),
]
for label, c1, c2, color in groups:
    ws.merge_cells(start_row=2, start_column=c1, end_row=2, end_column=c2)
    hc(ws, 2, c1, label, color)
ws.row_dimensions[2].height = 20

# Column headers
cols = list(df_res.columns)
col_bg = [C_H1]*4 + [C_H2]*5 + [C_H1]*2
for ci, (label, bg) in enumerate(zip(cols, col_bg), start=1):
    hc(ws, 3, ci, label, bg, sz=9, wrap=True)
ws.row_dimensions[3].height = 40

# Data rows
for ri, (_, row) in enumerate(df_res.iterrows(), start=4):
    station = row["Station"]
    bg_st   = station_colors.get(station, "FFFFFF")
    sig     = row["Significance"]

    # Row background based on significance
    if sig == "**":
        bg_row = C_SIG1
    elif sig == "*":
        bg_row = C_SIG2
    else:
        bg_row = "FFFFFF"

    for ci, col in enumerate(cols, start=1):
        val = row[col]
        if pd.isna(val): val = "—"

        if col == "Station":
            dc(ws, ri, ci, val, bg=bg_st, bold=True, align="left")
        elif col == "Index":
            dc(ws, ri, ci, val, bg=bg_st, align="center")
        elif col in ["Scale (month)","N (years)"]:
            dc(ws, ri, ci, val, bg=C_ALT if ri%2==0 else "FFFFFF")
        elif col == "Kendall τ":
            dc(ws, ri, ci, val, bg=bg_row, fmt="0.000")
        elif col == "Z statistic":
            dc(ws, ri, ci, val, bg=bg_row, fmt="+0.000;-0.000;0.000")
        elif col == "p-value":
            dc(ws, ri, ci, val, bg=bg_row, fmt="0.0000")
        elif col == "Significance":
            cell_bg = C_SIG1 if val=="**" else (C_SIG2 if val=="*" else C_NS)
            dc(ws, ri, ci, val, bg=cell_bg, bold=(val in ["*","**"]),
               align="center")
        elif col == "Sen's Slope":
            dc(ws, ri, ci, val, bg=bg_row, fmt="+0.00000;-0.00000;0.00000")
        elif col == "Trend":
            tr = row["Trend"]
            cell_bg = C_DEC if tr=="↓" else (C_INC if tr=="↑" else "FFFFFF")
            dc(ws, ri, ci, val, bg=cell_bg, bold=True)
        elif col == "Change Point Year":
            dc(ws, ri, ci, val if val != "—" else "—",
               bg=C_ALT if ri%2==0 else "FFFFFF",
               fmt="0" if val != "—" else None)
        else:
            dc(ws, ri, ci, val)

    # Station separator
    next_st = df_res.iloc[ri-4+1]["Station"] if ri-4+1 < len(df_res) else None
    if next_st != station:
        for ci in range(1, n_cols+1):
            c = ws.cell(row=ri, column=ci)
            c.border = Border(left=thin, right=thin,
                              top=thin, bottom=thick)

# Column widths
widths = [12, 7, 10, 8, 10, 11, 9, 11, 12, 8, 15]
for ci, w in enumerate(widths, start=1):
    ws.column_dimensions[get_column_letter(ci)].width = w

# Conditional formatting — color scale on Z statistic
last_row = 3 + len(df_res)
ws.conditional_formatting.add(
    f"F4:F{last_row}",
    ColorScaleRule(
        start_type="min", start_color="C6EFCE",
        mid_type="num",   mid_value=0, mid_color="FFFFFF",
        end_type="max",   end_color="FFC7CE"
    )
)

# Legend sheet
ws2 = wb.create_sheet("Legend")
legends = [
    ("**",  "p < 0.01 — Highly significant trend",       C_SIG1),
    ("*",   "p < 0.05 — Significant trend",              C_SIG2),
    ("ns",  "p ≥ 0.05 — No significant trend",           C_NS),
    ("↑",   "Increasing trend",                           C_INC),
    ("↓",   "Decreasing trend",                           C_DEC),
    ("—",   "No trend or insufficient data",              "FFFFFF"),
]
ws2.cell(row=1,column=1,value="Legend").font = Font(bold=True,size=11)
for ri,(sym,desc,col) in enumerate(legends,start=2):
    c1 = ws2.cell(row=ri,column=1,value=sym)
    c1.fill = PatternFill("solid",fgColor=col)
    c1.font = Font(bold=True,name="Calibri")
    c1.border = tb
    c2 = ws2.cell(row=ri,column=2,value=desc)
    c2.font = Font(name="Calibri",size=10)
    c2.border = tb
ws2.column_dimensions["A"].width = 8
ws2.column_dimensions["B"].width = 40

# Freeze & settings
ws.freeze_panes = "D4"
ws.sheet_view.showGridLines = False

out_path = os.path.join(OUTPUT_DIR, "Table4_MK_Trend.xlsx")
wb.save(out_path)
print(f"\nExcel saved → {out_path}")
print(f"Total rows  : {len(df_res)}")
print(f"\nTrend summary:")
print(df_res["Trend"].value_counts().to_string())
print(f"\nSignificance summary:")
print(df_res["Significance"].value_counts().to_string())

In [ ]:
"""
Table 4 — Sequential Mann-Kendall Trend Analysis Summary
Computes Z, p-value, Sen's slope, trend direction, and change point year
for all station × index × scale combinations.

Requirements:
    pip install pandas numpy scipy pymannkendall openpyxl xlsxwriter
"""

import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy.stats import kendalltau
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
INPUT_PATH = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari"

STATION_FILES = {
    "Kastamonu" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kastamonu_output_indices",
    "Sivas"     : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Sivas_output_indices",
    "Kayseri"   : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kayseri_output_indices",
    "Yozgat"    : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_output_indices",
    "Kirikkale" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kirikkale_output_indices",
}

DATE_COL   = "Date"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Tables"
INDEX_COLS = ["SPI-3","SPI-6","SPI-12","SDI-3","SDI-6","SDI-12","RDI-3","RDI-6","RDI-12"]
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ----------------------------------------------------------
# Load station data
# ----------------------------------------------------------
def load_station(path, date_col):
    if os.path.isdir(path):
        files = sorted([
            os.path.join(path, f) for f in os.listdir(path)
            if f.endswith((".xlsx",".xls",".csv")) and not f.startswith("~")
        ])
        frames = []
        seen   = set()
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            tmp = pd.read_excel(fp, parse_dates=[date_col]) if ext in [".xlsx",".xls"] \
                  else pd.read_csv(fp, parse_dates=[date_col])
            new_cols = [date_col] + [c for c in tmp.columns
                                     if c != date_col and c not in seen]
            tmp = tmp[new_cols]
            seen.update(tmp.columns)
            frames.append(tmp)
        df = frames[0]
        for f in frames[1:]:
            df = pd.merge(df, f, on=date_col, how="outer")
    else:
        ext = os.path.splitext(path)[1].lower()
        df  = pd.read_excel(path, parse_dates=[date_col]) if ext in [".xlsx",".xls"] \
              else pd.read_csv(path, parse_dates=[date_col])
    return df.sort_values(date_col).reset_index(drop=True)


# ----------------------------------------------------------
# Change point detection from SQMK (UF/UB intersection)
# ----------------------------------------------------------
def sequential_mk_changepoint(x):
    """Returns change point year index (None if not found)."""
    n  = len(x)
    uf = np.zeros(n)
    for i in range(1, n):
        s     = sum(np.sign(x[i] - x[j]) for j in range(i))
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        uf[i] = (s-e_s)/np.sqrt(var_s) if var_s > 0 else 0.0

    x_rev  = x[::-1]
    uf_rev = np.zeros(n)
    for i in range(1, n):
        s     = sum(np.sign(x_rev[i] - x_rev[j]) for j in range(i))
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        uf_rev[i] = (s-e_s)/np.sqrt(var_s) if var_s > 0 else 0.0
    ub = -uf_rev[::-1]

    # Find intersection within ±1.96 band
    for i in range(1, n):
        d_prev = uf[i-1] - ub[i-1]
        d_curr = uf[i]   - ub[i]
        if d_prev * d_curr < 0:
            if abs(uf[i]) < 1.96 and abs(ub[i]) < 1.96:
                return i   # index of change point
    return None


# ----------------------------------------------------------
# Main computation
# ----------------------------------------------------------
print("="*60)
print("  Table 4 — Mann-Kendall Trend Analysis")
print("="*60 + "\n")

records = []

for station, path in STATION_FILES.items():
    print(f">>> {station}")
    try:
        df = load_station(path, DATE_COL)
    except Exception as e:
        print(f"  [ERROR] {e}"); continue

    # Annual means for trend analysis
    df_y = df.set_index(DATE_COL).resample("YE").mean().dropna(how="all")
    years = df_y.index.year.values

    for col in INDEX_COLS:
        if col not in df_y.columns:
            print(f"  [SKIP] {col} not found"); continue

        col_series = df_y[col].dropna()
        series = col_series.values
        years_col = col_series.index.year.values   # years aligned with series
        if len(series) < 5:
            continue

        idx_name = col.split("-")[0]
        scale    = int(col.split("-")[1])

        # pymannkendall
        try:
            res = mk.original_test(series)
            z_stat    = round(res.z,      3)
            p_val     = round(res.p,      4)
            slope     = round(res.slope,  5)
            tau       = round(res.Tau,    3)
            trend_dir = res.trend   # "increasing" / "decreasing" / "no trend"
        except Exception:
            z_stat = p_val = slope = tau = np.nan
            trend_dir = "N/A"

        # Change point year (using years aligned to non-NaN series)
        cp_idx = sequential_mk_changepoint(series)
        if cp_idx is not None and cp_idx < len(years_col):
            cp_year = int(years_col[cp_idx])
        else:
            cp_year = np.nan

        # Significance label
        if pd.isna(p_val):
            sig = "N/A"
        elif p_val < 0.01:
            sig = "**"
        elif p_val < 0.05:
            sig = "*"
        else:
            sig = "ns"

        # Trend symbol
        if trend_dir == "increasing":
            trend_sym = "↑"
        elif trend_dir == "decreasing":
            trend_sym = "↓"
        else:
            trend_sym = "—"

        records.append({
            "Station"         : station,
            "Index"           : idx_name,
            "Scale (month)"   : scale,
            "N (years)"       : len(series),
            "Kendall τ"       : tau,
            "Z statistic"     : z_stat,
            "p-value"         : p_val,
            "Significance"    : sig,
            "Sen's Slope"     : slope,
            "Trend"           : trend_sym,
            "Change Point Year": cp_year,
        })

        print(f"  {col}: Z={z_stat:+.3f}  p={p_val:.4f} {sig}  "
              f"slope={slope:.5f}  trend={trend_sym}  CP={cp_year}")

    print()

df_res = pd.DataFrame(records)
df_res = df_res.sort_values(["Station","Index","Scale (month)"]).reset_index(drop=True)

# ----------------------------------------------------------
# Save CSV
# ----------------------------------------------------------
csv_path = os.path.join(OUTPUT_DIR, "Table4_MK_Trend.csv")
df_res.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}")

# ----------------------------------------------------------
# Publication-ready Excel
# ----------------------------------------------------------
wb  = Workbook()
ws  = wb.active
ws.title = "Table 4 – MK Trend Analysis"

thin  = Side(style="thin",   color="000000")
thick = Side(style="medium", color="000000")
tb    = Border(left=thin, right=thin, top=thin, bottom=thin)

C_H1   = "1F3864"   # dark navy
C_H2   = "2E75B6"   # mid blue
C_ALT  = "EBF3FB"   # light blue
C_SIG1 = "C6EFCE"   # green — significant *
C_SIG2 = "FFEB9C"   # yellow — near significant
C_NS   = "FFFFFF"   # white — not significant
C_INC  = "FFC7CE"   # red tint — increasing trend
C_DEC  = "C6EFCE"   # green tint — decreasing trend

station_colors = {
    "Kastamonu": "E8F4FD", "Kayseri":  "FEF9E7",
    "Kirikkale": "EAFAF1", "Sivas":    "FDEDEC",
    "Yozgat":   "F4ECF7",
}

def hc(ws, r, c, val, bg, fg="FFFFFF", bold=True, align="center",
        wrap=False, sz=10):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(bold=bold, color=fg, size=sz, name="Calibri")
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal=align, vertical="center",
                               wrap_text=wrap)
    cell.border    = tb
    return cell

def dc(ws, r, c, val, bg="FFFFFF", bold=False, align="center", fmt=None):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(bold=bold, color="000000", size=9, name="Calibri")
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal=align, vertical="center")
    cell.border    = tb
    if fmt: cell.number_format = fmt
    return cell

# Title
n_cols = len(df_res.columns)
ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
t = ws.cell(row=1, column=1,
    value="Table 4. Sequential Mann-Kendall Trend Analysis Results — "
          "Kızılırmak Basin (Annual means of monthly drought indices)")
t.font      = Font(bold=True, size=11, name="Calibri")
t.fill      = PatternFill("solid", fgColor="D9E2F3")
t.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
ws.row_dimensions[1].height = 30

# Group headers
groups = [
    ("Station Info",    1,  4,  C_H1),
    ("MK Statistics",   5,  9,  C_H2),
    ("Trend Result",   10, 11,  C_H1),
]
for label, c1, c2, color in groups:
    ws.merge_cells(start_row=2, start_column=c1, end_row=2, end_column=c2)
    hc(ws, 2, c1, label, color)
ws.row_dimensions[2].height = 20

# Column headers
cols = list(df_res.columns)
col_bg = [C_H1]*4 + [C_H2]*5 + [C_H1]*2
for ci, (label, bg) in enumerate(zip(cols, col_bg), start=1):
    hc(ws, 3, ci, label, bg, sz=9, wrap=True)
ws.row_dimensions[3].height = 40

# Data rows
for ri, (_, row) in enumerate(df_res.iterrows(), start=4):
    station = row["Station"]
    bg_st   = station_colors.get(station, "FFFFFF")
    sig     = row["Significance"]

    # Row background based on significance
    if sig == "**":
        bg_row = C_SIG1
    elif sig == "*":
        bg_row = C_SIG2
    else:
        bg_row = "FFFFFF"

    for ci, col in enumerate(cols, start=1):
        val = row[col]
        if pd.isna(val): val = "—"

        if col == "Station":
            dc(ws, ri, ci, val, bg=bg_st, bold=True, align="left")
        elif col == "Index":
            dc(ws, ri, ci, val, bg=bg_st, align="center")
        elif col in ["Scale (month)","N (years)"]:
            dc(ws, ri, ci, val, bg=C_ALT if ri%2==0 else "FFFFFF")
        elif col == "Kendall τ":
            dc(ws, ri, ci, val, bg=bg_row, fmt="0.000")
        elif col == "Z statistic":
            dc(ws, ri, ci, val, bg=bg_row, fmt="+0.000;-0.000;0.000")
        elif col == "p-value":
            dc(ws, ri, ci, val, bg=bg_row, fmt="0.0000")
        elif col == "Significance":
            cell_bg = C_SIG1 if val=="**" else (C_SIG2 if val=="*" else C_NS)
            dc(ws, ri, ci, val, bg=cell_bg, bold=(val in ["*","**"]),
               align="center")
        elif col == "Sen's Slope":
            dc(ws, ri, ci, val, bg=bg_row, fmt="+0.00000;-0.00000;0.00000")
        elif col == "Trend":
            tr = row["Trend"]
            cell_bg = C_DEC if tr=="↓" else (C_INC if tr=="↑" else "FFFFFF")
            dc(ws, ri, ci, val, bg=cell_bg, bold=True)
        elif col == "Change Point Year":
            raw = row[col]
            if pd.isna(raw) or raw == "—":
                dc(ws, ri, ci, "—", bg=C_ALT if ri%2==0 else "FFFFFF")
            else:
                dc(ws, ri, ci, int(raw), bg=C_ALT if ri%2==0 else "FFFFFF",
                   fmt="0")
        else:
            dc(ws, ri, ci, val)

    # Station separator
    next_st = df_res.iloc[ri-4+1]["Station"] if ri-4+1 < len(df_res) else None
    if next_st != station:
        for ci in range(1, n_cols+1):
            c = ws.cell(row=ri, column=ci)
            c.border = Border(left=thin, right=thin,
                              top=thin, bottom=thick)

# Column widths
widths = [12, 7, 10, 8, 10, 11, 9, 11, 12, 8, 15]
for ci, w in enumerate(widths, start=1):
    ws.column_dimensions[get_column_letter(ci)].width = w

# Conditional formatting — color scale on Z statistic
last_row = 3 + len(df_res)
ws.conditional_formatting.add(
    f"F4:F{last_row}",
    ColorScaleRule(
        start_type="min", start_color="C6EFCE",
        mid_type="num",   mid_value=0, mid_color="FFFFFF",
        end_type="max",   end_color="FFC7CE"
    )
)

# Legend sheet
ws2 = wb.create_sheet("Legend")
legends = [
    ("**",  "p < 0.01 — Highly significant trend",       C_SIG1),
    ("*",   "p < 0.05 — Significant trend",              C_SIG2),
    ("ns",  "p ≥ 0.05 — No significant trend",           C_NS),
    ("↑",   "Increasing trend",                           C_INC),
    ("↓",   "Decreasing trend",                           C_DEC),
    ("—",   "No trend or insufficient data",              "FFFFFF"),
]
ws2.cell(row=1,column=1,value="Legend").font = Font(bold=True,size=11)
for ri,(sym,desc,col) in enumerate(legends,start=2):
    c1 = ws2.cell(row=ri,column=1,value=sym)
    c1.fill = PatternFill("solid",fgColor=col)
    c1.font = Font(bold=True,name="Calibri")
    c1.border = tb
    c2 = ws2.cell(row=ri,column=2,value=desc)
    c2.font = Font(name="Calibri",size=10)
    c2.border = tb
ws2.column_dimensions["A"].width = 8
ws2.column_dimensions["B"].width = 40

# Freeze & settings
ws.freeze_panes = "D4"
ws.sheet_view.showGridLines = False

out_path = os.path.join(OUTPUT_DIR, "Table4_MK_Trend.xlsx")
wb.save(out_path)
print(f"\nExcel saved → {out_path}")
print(f"Total rows  : {len(df_res)}")
print(f"\nTrend summary:")
print(df_res["Trend"].value_counts().to_string())
print(f"\nSignificance summary:")
print(df_res["Significance"].value_counts().to_string())


In [ ]:
"""
Table 4 — Sequential Mann-Kendall Trend Analysis Summary
Computes Z, p-value, Sen's slope, trend direction, and change point year
for all station × index × scale combinations.

Requirements:
    pip install pandas numpy scipy pymannkendall openpyxl xlsxwriter
"""

import numpy as np
import pandas as pd
import pymannkendall as mk
from scipy.stats import kendalltau
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
INPUT_PATH = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari"

STATION_FILES = {
    "Kastamonu" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kastamonu_output_indices",
    "Sivas"     : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Sivas_output_indices",
    "Kayseri"   : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kayseri_output_indices",
    "Yozgat"    : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_output_indices",
    "Kirikkale" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kirikkale_output_indices",
}

DATE_COL   = "Date"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Tables"
INDEX_COLS = ["SPI-3","SPI-6","SPI-12","SDI-3","SDI-6","SDI-12","RDI-3","RDI-6","RDI-12"]
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ----------------------------------------------------------
# Load station data
# ----------------------------------------------------------
def load_station(path, date_col):
    if os.path.isdir(path):
        files = sorted([
            os.path.join(path, f) for f in os.listdir(path)
            if f.endswith((".xlsx",".xls",".csv")) and not f.startswith("~")
        ])
        frames = []
        seen   = set()
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            tmp = pd.read_excel(fp, parse_dates=[date_col]) if ext in [".xlsx",".xls"] \
                  else pd.read_csv(fp, parse_dates=[date_col])
            new_cols = [date_col] + [c for c in tmp.columns
                                     if c != date_col and c not in seen]
            tmp = tmp[new_cols]
            seen.update(tmp.columns)
            frames.append(tmp)
        df = frames[0]
        for f in frames[1:]:
            df = pd.merge(df, f, on=date_col, how="outer")
    else:
        ext = os.path.splitext(path)[1].lower()
        df  = pd.read_excel(path, parse_dates=[date_col]) if ext in [".xlsx",".xls"] \
              else pd.read_csv(path, parse_dates=[date_col])
    return df.sort_values(date_col).reset_index(drop=True)


# ----------------------------------------------------------
# Change point detection from SQMK (UF/UB intersection)
# ----------------------------------------------------------
def sequential_mk_changepoint(x):
    """
    Sequential Mann-Kendall change point detection.
    UF: forward statistic
    UB: backward statistic (computed on reversed series, then reversed back,
        with sign flipped — standard formulation)
    Returns index of first UF/UB intersection (change point).
    """
    n  = len(x)

    # --- Forward (UF) ---
    s_fwd = np.zeros(n)
    uf    = np.zeros(n)
    for i in range(1, n):
        for j in range(i):
            s_fwd[i] += np.sign(x[i] - x[j])
        e_s   = i * (i - 1) / 4.0
        var_s = i * (i - 1) * (2*i + 5) / 72.0
        uf[i] = (s_fwd[i] - e_s) / np.sqrt(var_s) if var_s > 0 else 0.0

    # --- Backward (UB): run UF on reversed series, reverse result, negate ---
    x_rev  = x[::-1]
    s_bck  = np.zeros(n)
    ub_rev = np.zeros(n)
    for i in range(1, n):
        for j in range(i):
            s_bck[i] += np.sign(x_rev[i] - x_rev[j])
        e_s   = i * (i - 1) / 4.0
        var_s = i * (i - 1) * (2*i + 5) / 72.0
        ub_rev[i] = (s_bck[i] - e_s) / np.sqrt(var_s) if var_s > 0 else 0.0

    # UB: reverse ub_rev array (do NOT negate — standard formulation)
    ub = ub_rev[::-1]

    # --- Find first intersection of UF and UB ---
    for i in range(1, n - 1):
        if (uf[i] - ub[i]) * (uf[i+1] - ub[i+1]) <= 0:
            return i
    return None


# ----------------------------------------------------------
# Main computation
# ----------------------------------------------------------
print("="*60)
print("  Table 4 — Mann-Kendall Trend Analysis")
print("="*60 + "\n")

records = []

for station, path in STATION_FILES.items():
    print(f">>> {station}")
    try:
        df = load_station(path, DATE_COL)
    except Exception as e:
        print(f"  [ERROR] {e}"); continue

    # Annual means for trend analysis
    df_y = df.set_index(DATE_COL).resample("YE").mean().dropna(how="all")
    years = df_y.index.year.values

    for col in INDEX_COLS:
        if col not in df_y.columns:
            print(f"  [SKIP] {col} not found"); continue

        col_series = df_y[col].dropna()
        series = col_series.values
        years_col = col_series.index.year.values   # years aligned with series
        if len(series) < 5:
            continue

        idx_name = col.split("-")[0]
        scale    = int(col.split("-")[1])

        # pymannkendall
        try:
            res = mk.original_test(series)
            z_stat    = round(res.z,      3)
            p_val     = round(res.p,      4)
            slope     = round(res.slope,  5)
            tau       = round(res.Tau,    3)
            trend_dir = res.trend   # "increasing" / "decreasing" / "no trend"
        except Exception:
            z_stat = p_val = slope = tau = np.nan
            trend_dir = "N/A"

        # Change point year (using years aligned to non-NaN series)
        cp_idx = sequential_mk_changepoint(series)
        if cp_idx is not None and cp_idx < len(years_col):
            cp_year = int(years_col[cp_idx])
        else:
            cp_year = np.nan

        # Significance label
        if pd.isna(p_val):
            sig = "N/A"
        elif p_val < 0.01:
            sig = "**"
        elif p_val < 0.05:
            sig = "*"
        else:
            sig = "ns"

        # Trend symbol
        if trend_dir == "increasing":
            trend_sym = "↑"
        elif trend_dir == "decreasing":
            trend_sym = "↓"
        else:
            trend_sym = "—"

        records.append({
            "Station"         : station,
            "Index"           : idx_name,
            "Scale (month)"   : scale,
            "N (years)"       : len(series),
            "Kendall τ"       : tau,
            "Z statistic"     : z_stat,
            "p-value"         : p_val,
            "Significance"    : sig,
            "Sen's Slope"     : slope,
            "Trend"           : trend_sym,
            "Change Point Year": cp_year,
        })

        print(f"  {col}: Z={z_stat:+.3f}  p={p_val:.4f} {sig}  "
              f"slope={slope:.5f}  trend={trend_sym}  CP={cp_year}")

    print()

df_res = pd.DataFrame(records)
df_res = df_res.sort_values(["Station","Index","Scale (month)"]).reset_index(drop=True)

# ----------------------------------------------------------
# Save CSV
# ----------------------------------------------------------
csv_path = os.path.join(OUTPUT_DIR, "Table4_MK_Trend.csv")
df_res.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}")

# ----------------------------------------------------------
# Publication-ready Excel
# ----------------------------------------------------------
wb  = Workbook()
ws  = wb.active
ws.title = "Table 4 – MK Trend Analysis"

thin  = Side(style="thin",   color="000000")
thick = Side(style="medium", color="000000")
tb    = Border(left=thin, right=thin, top=thin, bottom=thin)

C_H1   = "1F3864"   # dark navy
C_H2   = "2E75B6"   # mid blue
C_ALT  = "EBF3FB"   # light blue
C_SIG1 = "C6EFCE"   # green — significant *
C_SIG2 = "FFEB9C"   # yellow — near significant
C_NS   = "FFFFFF"   # white — not significant
C_INC  = "FFC7CE"   # red tint — increasing trend
C_DEC  = "C6EFCE"   # green tint — decreasing trend

station_colors = {
    "Kastamonu": "E8F4FD", "Kayseri":  "FEF9E7",
    "Kirikkale": "EAFAF1", "Sivas":    "FDEDEC",
    "Yozgat":   "F4ECF7",
}

def hc(ws, r, c, val, bg, fg="FFFFFF", bold=True, align="center",
        wrap=False, sz=10):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(bold=bold, color=fg, size=sz, name="Calibri")
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal=align, vertical="center",
                               wrap_text=wrap)
    cell.border    = tb
    return cell

def dc(ws, r, c, val, bg="FFFFFF", bold=False, align="center", fmt=None):
    cell = ws.cell(row=r, column=c, value=val)
    cell.font      = Font(bold=bold, color="000000", size=9, name="Calibri")
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal=align, vertical="center")
    cell.border    = tb
    if fmt: cell.number_format = fmt
    return cell

# Title
n_cols = len(df_res.columns)
ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
t = ws.cell(row=1, column=1,
    value="Table 4. Sequential Mann-Kendall Trend Analysis Results — "
          "Kızılırmak Basin (Annual means of monthly drought indices)")
t.font      = Font(bold=True, size=11, name="Calibri")
t.fill      = PatternFill("solid", fgColor="D9E2F3")
t.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
ws.row_dimensions[1].height = 30

# Group headers
groups = [
    ("Station Info",    1,  4,  C_H1),
    ("MK Statistics",   5,  9,  C_H2),
    ("Trend Result",   10, 11,  C_H1),
]
for label, c1, c2, color in groups:
    ws.merge_cells(start_row=2, start_column=c1, end_row=2, end_column=c2)
    hc(ws, 2, c1, label, color)
ws.row_dimensions[2].height = 20

# Column headers
cols = list(df_res.columns)
col_bg = [C_H1]*4 + [C_H2]*5 + [C_H1]*2
for ci, (label, bg) in enumerate(zip(cols, col_bg), start=1):
    hc(ws, 3, ci, label, bg, sz=9, wrap=True)
ws.row_dimensions[3].height = 40

# Data rows
for ri, (_, row) in enumerate(df_res.iterrows(), start=4):
    station = row["Station"]
    bg_st   = station_colors.get(station, "FFFFFF")
    sig     = row["Significance"]

    # Row background based on significance
    if sig == "**":
        bg_row = C_SIG1
    elif sig == "*":
        bg_row = C_SIG2
    else:
        bg_row = "FFFFFF"

    for ci, col in enumerate(cols, start=1):
        val = row[col]
        if pd.isna(val): val = "—"

        if col == "Station":
            dc(ws, ri, ci, val, bg=bg_st, bold=True, align="left")
        elif col == "Index":
            dc(ws, ri, ci, val, bg=bg_st, align="center")
        elif col in ["Scale (month)","N (years)"]:
            dc(ws, ri, ci, val, bg=C_ALT if ri%2==0 else "FFFFFF")
        elif col == "Kendall τ":
            dc(ws, ri, ci, val, bg=bg_row, fmt="0.000")
        elif col == "Z statistic":
            dc(ws, ri, ci, val, bg=bg_row, fmt="+0.000;-0.000;0.000")
        elif col == "p-value":
            dc(ws, ri, ci, val, bg=bg_row, fmt="0.0000")
        elif col == "Significance":
            cell_bg = C_SIG1 if val=="**" else (C_SIG2 if val=="*" else C_NS)
            dc(ws, ri, ci, val, bg=cell_bg, bold=(val in ["*","**"]),
               align="center")
        elif col == "Sen's Slope":
            dc(ws, ri, ci, val, bg=bg_row, fmt="+0.00000;-0.00000;0.00000")
        elif col == "Trend":
            tr = row["Trend"]
            cell_bg = C_DEC if tr=="↓" else (C_INC if tr=="↑" else "FFFFFF")
            dc(ws, ri, ci, val, bg=cell_bg, bold=True)
        elif col == "Change Point Year":
            raw = row[col]
            if pd.isna(raw) or raw == "—":
                dc(ws, ri, ci, "—", bg=C_ALT if ri%2==0 else "FFFFFF")
            else:
                dc(ws, ri, ci, int(raw), bg=C_ALT if ri%2==0 else "FFFFFF",
                   fmt="0")
        else:
            dc(ws, ri, ci, val)

    # Station separator
    next_st = df_res.iloc[ri-4+1]["Station"] if ri-4+1 < len(df_res) else None
    if next_st != station:
        for ci in range(1, n_cols+1):
            c = ws.cell(row=ri, column=ci)
            c.border = Border(left=thin, right=thin,
                              top=thin, bottom=thick)

# Column widths
widths = [12, 7, 10, 8, 10, 11, 9, 11, 12, 8, 15]
for ci, w in enumerate(widths, start=1):
    ws.column_dimensions[get_column_letter(ci)].width = w

# Conditional formatting — color scale on Z statistic
last_row = 3 + len(df_res)
ws.conditional_formatting.add(
    f"F4:F{last_row}",
    ColorScaleRule(
        start_type="min", start_color="C6EFCE",
        mid_type="num",   mid_value=0, mid_color="FFFFFF",
        end_type="max",   end_color="FFC7CE"
    )
)

# Legend sheet
ws2 = wb.create_sheet("Legend")
legends = [
    ("**",  "p < 0.01 — Highly significant trend",       C_SIG1),
    ("*",   "p < 0.05 — Significant trend",              C_SIG2),
    ("ns",  "p ≥ 0.05 — No significant trend",           C_NS),
    ("↑",   "Increasing trend",                           C_INC),
    ("↓",   "Decreasing trend",                           C_DEC),
    ("—",   "No trend or insufficient data",              "FFFFFF"),
]
ws2.cell(row=1,column=1,value="Legend").font = Font(bold=True,size=11)
for ri,(sym,desc,col) in enumerate(legends,start=2):
    c1 = ws2.cell(row=ri,column=1,value=sym)
    c1.fill = PatternFill("solid",fgColor=col)
    c1.font = Font(bold=True,name="Calibri")
    c1.border = tb
    c2 = ws2.cell(row=ri,column=2,value=desc)
    c2.font = Font(name="Calibri",size=10)
    c2.border = tb
ws2.column_dimensions["A"].width = 8
ws2.column_dimensions["B"].width = 40

# Freeze & settings
ws.freeze_panes = "D4"
ws.sheet_view.showGridLines = False

out_path = os.path.join(OUTPUT_DIR, "Table4_MK_Trend.xlsx")
wb.save(out_path)
print(f"\nExcel saved → {out_path}")
print(f"Total rows  : {len(df_res)}")
print(f"\nTrend summary:")
print(df_res["Trend"].value_counts().to_string())
print(f"\nSignificance summary:")
print(df_res["Significance"].value_counts().to_string())

In [ ]:
pip install pandas numpy matplotlib pymannkendall openpyxl

In [ ]:
"""
Figure 4 — Sequential Mann-Kendall Trend Analysis
9 PNG: one per index-scale combination (SPI-3, SPI-6, SPI-12,
       SDI-3, SDI-6, SDI-12, RDI-3, RDI-6, RDI-12)
Each figure: 5 stations as subplots in a single row

Requirements:
    pip install pandas numpy matplotlib pymannkendall openpyxl
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import pymannkendall as mk
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
STATION_FILES = {
    "Kastamonu" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kastamonu_output_indices",
    "Sivas"     : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Sivas_output_indices",
    "Kayseri"   : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kayseri_output_indices",
    "Yozgat"    : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_output_indices",
    "Kirikkale" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kirikkale_output_indices",
}
DATE_COL   = "Date"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/SQMK_Figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

INDICES = ["SPI", "SDI", "RDI"]
SCALES  = [3, 6, 12]

# Color per index
IDX_COLOR = {"SPI": "#1f6f2e", "SDI": "#08519c", "RDI": "#8c2d04"}

# Bar colors: positive = dark, negative = light version
BAR_POS = {"SPI": "#2e7d32", "SDI": "#1565c0", "RDI": "#bf360c"}
BAR_NEG = {"SPI": "#a5d6a7", "SDI": "#90caf9", "RDI": "#ffab91"}

# ============================================================
# LOAD DATA
# ============================================================
def load_station(path, date_col):
    if os.path.isdir(path):
        files = sorted([
            os.path.join(path, f) for f in os.listdir(path)
            if f.endswith((".xlsx",".xls",".csv")) and not f.startswith("~")
        ])
        seen = set(); frames = []
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            tmp = pd.read_excel(fp, parse_dates=[date_col]) \
                  if ext in [".xlsx",".xls"] \
                  else pd.read_csv(fp, parse_dates=[date_col])
            new_cols = [date_col]+[c for c in tmp.columns
                                   if c != date_col and c not in seen]
            tmp = tmp[new_cols]; seen.update(tmp.columns)
            frames.append(tmp)
        df = frames[0]
        for f in frames[1:]:
            df = pd.merge(df, f, on=date_col, how="outer")
    else:
        ext = os.path.splitext(path)[1].lower()
        df  = pd.read_excel(path, parse_dates=[date_col]) \
              if ext in [".xlsx",".xls"] \
              else pd.read_csv(path, parse_dates=[date_col])
    return df.sort_values(date_col).reset_index(drop=True)


print("Loading station data...")
STATION_DATA = {}
for station, path in STATION_FILES.items():
    try:
        STATION_DATA[station] = load_station(path, DATE_COL)
        print(f"  {station}: {len(STATION_DATA[station])} rows")
    except Exception as e:
        print(f"  [ERROR] {station}: {e}")

# ============================================================
# SEQUENTIAL MK
# ============================================================
def sequential_mk(x):
    n = len(x)
    # UF (forward)
    uf = np.zeros(n)
    for i in range(1, n):
        s = 0
        for j in range(i):
            s += np.sign(x[i] - x[j])
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        uf[i] = (s - e_s) / np.sqrt(var_s) if var_s > 0 else 0.0
    # UB (backward)
    x_rev = x[::-1]
    ub_rev = np.zeros(n)
    for i in range(1, n):
        s = 0
        for j in range(i):
            s += np.sign(x_rev[i] - x_rev[j])
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        ub_rev[i] = (s - e_s) / np.sqrt(var_s) if var_s > 0 else 0.0
    ub = ub_rev[::-1]
    return uf, ub


def find_changepoint(uf, ub):
    for i in range(1, len(uf)-1):
        if (uf[i]-ub[i])*(uf[i+1]-ub[i+1]) <= 0:
            return i
    return None


# ============================================================
# PLOT ONE PANEL (one station)
# ============================================================
def plot_one_station(ax, ax2, station, col, idx, scale):
    df = STATION_DATA.get(station)
    if df is None or col not in df.columns:
        ax.text(0.5, 0.5, "No data", ha="center", va="center",
                transform=ax.transAxes, fontsize=10, color="gray")
        return

    # Annual mean
    series = (df.set_index(DATE_COL)[col]
                .resample("YE").mean()
                .dropna())
    if len(series) < 5:
        ax.text(0.5, 0.5, f"n={len(series)}\n(insufficient)",
                ha="center", va="center",
                transform=ax.transAxes, fontsize=9, color="gray")
        return

    years  = series.index.year.values
    values = series.values
    n      = len(values)
    t      = np.arange(n)

    # ── Bar chart (annual index values) ──────────────────
    bar_colors = [BAR_POS[idx] if v >= 0 else BAR_NEG[idx] for v in values]
    ax.bar(years, values, color=bar_colors, alpha=0.80,
           width=0.8, zorder=2)
    ax.axhline(0, color="black", linewidth=0.7, zorder=3)
    ax.set_xlim(years[0]-0.8, years[-1]+0.8)
    yabs = max(abs(values.min()), abs(values.max()))
    ax.set_ylim(-yabs*1.35, yabs*1.35)
    ax.grid(axis="y", linestyle="--", alpha=0.3, zorder=1)
    ax.tick_params(axis="both", labelsize=7)
    ax.set_ylabel(f"{col}", fontsize=8, color="black")
    ax.yaxis.label.set_color("black")

    # ── SMK statistics ────────────────────────────────────
    uf, ub = sequential_mk(values)

    mk_range = max(abs(uf).max(), abs(ub).max(), 2.5) * 1.25

    ax2.plot(years, uf, color="navy",    linewidth=1.8,
             linestyle="-",  label="UF (Forward)", zorder=5)
    ax2.plot(years, ub, color="darkred", linewidth=1.5,
             linestyle="--", label="UB (Backward)", zorder=5)
    ax2.axhline( 1.96, color="black", linewidth=0.9,
                 linestyle=":", label="95% CI", zorder=4)
    ax2.axhline(-1.96, color="black", linewidth=0.9,
                 linestyle=":", zorder=4)
    ax2.set_ylim(-mk_range, mk_range)
    ax2.set_xlim(years[0]-0.8, years[-1]+0.8)
    ax2.tick_params(axis="y", labelsize=7, labelcolor="navy")
    ax2.set_ylabel("MK Statistic (u)", fontsize=7.5, color="navy")
    ax2.spines["right"].set_edgecolor("navy")
    ax2.spines["right"].set_linewidth(1.2)

    # Change point marker
    cp_idx = find_changepoint(uf, ub)
    if cp_idx is not None:
        cp_year = years[cp_idx]
        ax2.axvline(cp_year, color="darkorange", linewidth=1.2,
                    linestyle="-.", alpha=0.8, zorder=6)
        ax2.text(cp_year+0.3, mk_range*0.88,
                 f"{cp_year}", fontsize=7,
                 color="darkorange", fontweight="bold")

    # MK test summary
    try:
        res    = mk.original_test(values)
        trend  = res.trend
        pval   = res.p
        slope  = res.slope
        sig    = "**" if pval < 0.01 else ("*" if pval < 0.05 else "ns")
        summary = f"Z={res.z:+.2f}  p={pval:.3f}{sig}\nslope={slope:+.4f}"
    except Exception:
        summary = "MK test failed"

    ax.set_title(station, fontsize=9, fontweight="bold", pad=4)
    ax.text(0.98, 0.02, summary,
            transform=ax.transAxes,
            ha="right", va="bottom", fontsize=6.5,
            color="#333333",
            bbox=dict(boxstyle="round,pad=0.3",
                      facecolor="white", alpha=0.8,
                      edgecolor="lightgray"))


# ============================================================
# MAKE FIGURE FOR ONE INDEX-SCALE
# ============================================================
def make_figure4(idx, scale):
    col = f"{idx}-{scale}"
    stations = list(STATION_FILES.keys())
    n_st = len(stations)

    fig = plt.figure(figsize=(4.5*n_st, 6))
    fig.patch.set_facecolor("white")

    # Two rows of axes per station: top=bar, bottom=MK
    # Use gridspec with twin axes approach
    outer = gridspec.GridSpec(1, n_st, figure=fig,
                              wspace=0.30, left=0.06, right=0.97,
                              top=0.88, bottom=0.12)

    for si, station in enumerate(stations):
        inner = gridspec.GridSpecFromSubplotSpec(
            2, 1,
            subplot_spec=outer[si],
            hspace=0.05,
            height_ratios=[1, 1],
        )
        ax_bar = fig.add_subplot(inner[0])
        ax_mk  = fig.add_subplot(inner[1])
        ax2    = ax_mk.twinx()

        # Bar chart (annual values)
        df = STATION_DATA.get(station)
        if df is None or col not in df.columns:
            ax_bar.text(0.5, 0.5, "No data",
                        ha="center", va="center",
                        transform=ax_bar.transAxes, color="gray")
            ax_bar.set_title(station, fontsize=9, fontweight="bold")
            continue

        series = (df.set_index(DATE_COL)[col]
                    .resample("YE").mean()
                    .dropna())
        if len(series) < 5:
            ax_bar.text(0.5, 0.5, "Insufficient data",
                        ha="center", va="center",
                        transform=ax_bar.transAxes, color="gray")
            ax_bar.set_title(station, fontsize=9, fontweight="bold")
            continue

        years  = series.index.year.values
        values = series.values
        n      = len(values)

        # ── Panel 1: Annual bar ──────────────────────────
        bar_colors = [BAR_POS[idx] if v >= 0
                      else BAR_NEG[idx] for v in values]
        ax_bar.bar(years, values, color=bar_colors,
                   alpha=0.82, width=0.8, zorder=2)
        ax_bar.axhline(0, color="black", linewidth=0.7, zorder=3)
        yabs = max(abs(values.min()), abs(values.max()))
        ax_bar.set_ylim(-yabs*1.35, yabs*1.35)
        ax_bar.set_xlim(years[0]-0.8, years[-1]+0.8)
        ax_bar.grid(axis="y", linestyle="--", alpha=0.3)
        ax_bar.tick_params(labelbottom=False, labelsize=7)
        ax_bar.set_title(station, fontsize=9,
                         fontweight="bold", pad=3)
        if si == 0:
            ax_bar.set_ylabel(f"{col}\n(Annual Mean)",
                              fontsize=8)

        # MK test result text
        try:
            res   = mk.original_test(values)
            sig   = "**" if res.p < 0.01 else \
                    ("*" if res.p < 0.05 else "ns")
            txt   = (f"Z={res.z:+.2f}  {sig}\n"
                     f"slope={res.slope:+.4f}")
        except Exception:
            txt = ""

        ax_bar.text(0.98, 0.04, txt,
                    transform=ax_bar.transAxes,
                    ha="right", va="bottom", fontsize=6.5,
                    color="#222222",
                    bbox=dict(boxstyle="round,pad=0.25",
                              facecolor="white", alpha=0.85,
                              edgecolor="lightgray"))

        # ── Panel 2: UF / UB ────────────────────────────
        uf, ub = sequential_mk(values)
        mk_range = max(abs(uf).max(), abs(ub).max(), 2.5) * 1.2

        ax_mk.fill_between(years,  1.96, -1.96,
                           color="lightyellow", alpha=0.5, zorder=1)
        ax_mk.axhline( 1.96, color="black",
                       linewidth=0.8, linestyle=":", zorder=3)
        ax_mk.axhline(-1.96, color="black",
                       linewidth=0.8, linestyle=":", zorder=3)
        ax_mk.axhline(0, color="black",
                      linewidth=0.5, zorder=3)

        ax_mk.plot(years, uf, color="navy",    linewidth=1.8,
                   linestyle="-",  label="UF", zorder=5)
        ax_mk.plot(years, ub, color="darkred", linewidth=1.5,
                   linestyle="--", label="UB", zorder=5)

        ax_mk.set_ylim(-mk_range, mk_range)
        ax_mk.set_xlim(years[0]-0.8, years[-1]+0.8)
        ax_mk.tick_params(axis="x", labelsize=7, rotation=45)
        ax_mk.tick_params(axis="y", labelsize=7)
        ax_mk.grid(axis="y", linestyle="--", alpha=0.25)
        if si == 0:
            ax_mk.set_ylabel("MK Statistic (u)",
                             fontsize=8)

        # Change point
        cp_idx = find_changepoint(uf, ub)
        if cp_idx is not None and cp_idx < len(years):
            cp_year = years[cp_idx]
            ax_mk.axvline(cp_year, color="darkorange",
                          linewidth=1.3, linestyle="-.",
                          alpha=0.9, zorder=6)
            ax_mk.text(cp_year+0.5, mk_range*0.82,
                       str(cp_year), fontsize=6.5,
                       color="darkorange", fontweight="bold")

        # Legend only on first station
        if si == 0:
            ax_mk.legend(fontsize=7, loc="lower left",
                         framealpha=0.8)

    # ── Figure title & legend ────────────────────────────
    trend_sym = {"SPI": "Meteorological", "SDI": "Hydrological",
                 "RDI": "Reconnaissance"}
    fig.suptitle(
        f"Figure 4. Sequential Mann-Kendall Analysis — "
        f"{trend_sym[idx]} Drought Index ({col})\n"
        f"Top: Annual mean values  |  Bottom: UF (forward) & UB (backward) statistics  |  "
        f"Dotted lines: ±1.96 (95% CI)  |  Orange dashed: change point",
        fontsize=10, fontweight="bold"
    )

    out = os.path.join(OUTPUT_DIR, f"Figure4_SQMK_{col}.png")
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out}")


# ============================================================
# MAIN
# ============================================================
print("\n" + "="*55)
print("  Figure 4 — Sequential Mann-Kendall Trend Plots")
print("="*55 + "\n")

for idx in INDICES:
    for scale in SCALES:
        print(f">>> {idx}-{scale}")
        make_figure4(idx, scale)

print("\nAll 9 figures completed!")
print(f"Output → {OUTPUT_DIR}")